# Importações

In [1]:
import sys
import re
import os
import pandas as pd
import unicodedata
from pyspark.sql import SparkSession
# from awsglue.transforms import *
# from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
# from awsglue.context import GlueContext
# from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Funções

In [2]:
def ler_df_csv(sessao_spark, caminho):
    df = (
     sessao_spark.read
     .format("csv")
     .option("header", "true")
     .option("inferSchema", "false")
     .option("multiLine", "true")
     .option("quote", '"')
     .option("escape", '"')
     .option("encoding", "UTF-8")
     .load(caminho)
      )
    return df

def normalizar_coluna(coluna):
    if "', '" in coluna:
        coluna = coluna.split("', '", 1)[1]
    coluna = coluna.replace("('", "").replace("')", "").replace('"', '').strip()
    coluna = unicodedata.normalize('NFKD', coluna).encode('ascii', 'ignore').decode('utf-8')
    coluna = coluna.lower()
    coluna = re.sub(r"[^a-z0-9_]+", "_", coluna)
    coluna = re.sub(r"_+", "_", coluna).strip("_")
    if coluna and coluna[0].isdigit():
        coluna = "_" + coluna
    return coluna

def normalizar_valor_string_data(valor):
    if valor is None: return None
    if not isinstance(valor, str): return valor
    v = unicodedata.normalize('NFKD', valor).encode('ascii', 'ignore').decode('utf-8')
    v = re.sub(r'\s+', '_', v)
    return re.sub(r'__+', '_', v).strip('_')

udf_normalizar_valor_string_data = udf(normalizar_valor_string_data, StringType())

def renomear_colunas(df, colunas_map):
    cols_atuais = df.columns
    mapeamento_final = []
    nomes_utilizados = {}

    for c in cols_atuais:
        norm_c = normalizar_coluna(c)
        novo_nome = c
        for chave_mapa, valor_mapa in colunas_map.items():
            if norm_c == normalizar_coluna(chave_mapa) or c == chave_mapa:
                novo_nome = valor_mapa
                break

        if novo_nome in nomes_utilizados:
            nomes_utilizados[novo_nome] += 1
            novo_nome = f"{novo_nome}_{nomes_utilizados[novo_nome]}"
        else:
            nomes_utilizados[novo_nome] = 0

        mapeamento_final.append(novo_nome)

    processados = []
    for i in range(len(cols_atuais)):
        # O uso de F.col com crases garante que caracteres especiais no nome original não causem erro de resolução
        processados.append(F.col(f"`{cols_atuais[i]}`").alias(mapeamento_final[i]))

    return df.select(*processados)

def exportar_df_para_csv(df_spark, nome_arquivo):
    df_spark.toPandas().to_csv(nome_arquivo, index=False)

def processar_dataframe(df_spark):
    new_cols = []
    for i, (col_name, col_type) in enumerate(df_spark.dtypes):
        col_ref = df_spark[i]
        if col_type == 'string':
            new_cols.append(udf_normalizar_valor_string_data(col_ref).alias(col_name))
        else:
            new_cols.append(col_ref.alias(col_name))
    return df_spark.select(*new_cols)

# Iniciando projeto

In [3]:
sc = SparkContext.getOrCreate()
# glueContext = GlueContext(sc)
# spark = glueContext.spark_session
sessao = SparkSession.builder.appName("Bronze").getOrCreate()
# job = Job(glueContext)

In [4]:
novos_nomes_colunas_2023 = {
    "id": "id",
    "Idade": "idade",
    "Faixa_idade": "faixa_etaria",
    "Genero": "genero",
    "Cor_raca_etnia": "cor_raca_etnia",
    "PCD": "pcd",
    "experiencia_profissional_prejudicada": "exp_prof_prejud",
    "Nao_acredito_que_minha_experiencia_profissional_seja_afetada": "exp_prof_nao_prejud",
    "Experiencia_prejudicada_devido_a_minha_Cor_Raca_Etnia": "exp_prof_prejud_cor_raca_etnia",
    "Experiencia_prejudicada_devido_a_minha_identidade_de_genero": "exp_prof_prejud_ident_gen",
    "Experiencia_prejudicada_devido_ao_fato_de_ser_PCD": "exp_prof_prejud_pcd",
    "aspectos_prejudicados": "aspectos_prejud",
    "Quantidade_de_oportunidades_de_emprego_vagas_recebidas": "qtd_oportunidades_vagas_emprego_receb",
    "Senioridade_das_vagas_recebidas_em_relacao_a_sua_experiencia": "senioridade_vagas_recebidas_rel_experiencia",
    "Aprovacao_em_processos_seletivos_entrevistas": "aprov_processos_seletivos_entrevistas",
    "Oportunidades_de_progressao_de_carreira": "oportunidades_progresso_carreira",
    "Velocidade_de_progressao_de_carreira": "vel_progressao_carreira",
    "Nivel_de_cobranca_no_trabalho_Stress_no_trabalho": "nvl_cobranca_e_stress_trab",
    "Atencao_dada_diante_das_minhas_opinioes_e_ideias": "atencao_opiniao_ideias",
    "Relacao_com_outros_membros_da_empresa_em_momentos_de_trabalho": "rel_outros_membros_empresa_em_trabalho",
    "Relacao_com_outros_membros_da_empresa_em_momentos_de_integracao_e_outros_momentos_fora_do_trabalho": "rel_outros_membros_empresa_integracao_fora_trab",
    "vive_no_brasil": "vive_brasil",
    "Estado_onde_mora": "estado",
    "uf_onde_mora": "uf",
    "Regiao_onde_mora": "regiao",
    "Mudou_de_Estado": "mudou_estado",
    "Regiao_de_origem": "regiao_origem",
    "Nivel_de_Ensino": "nvl_ensino",
    "Area_de_Formacao": "area_formacao",
    "Qual_sua_situacao_atual_de_trabalho": "sit_atual_trab",
    "Setor": "setor",
    "Numero_de_Funcionarios": "n_funcionarios",
    "Gestor": "gestor",
    "Cargo_como_Gestor": "cargo_gestor",
    "Cargo_Atual": "cargo_atual",
    "Nivel": "nvl",
    "Faixa_salarial": "faixa_salarial",
    "Quanto_tempo_de_experiencia_na_area_de_dados_voce_tem": "tempo_exp_dados",
    "Quanto_tempo_de_experiencia_na_area_de_TI_Engenharia_de_Software_voce_teve_antes_de_comecar_a_trabalhar_na_area_de_dados": "tempo_exp_ti",
    "Voce_esta_satisfeito_na_sua_empresa_atual": "satisfacao_empresa_atual",
    "Qual_o_principal_motivo_da_sua_insatisfacao_com_a_empresa_atual": "motivo_insatis_empresa_atual",
    "Falta_de_oportunidade_de_crescimento_no_emprego_atual": "falta_oportunidade_cresc",
    "Salario_atual_nao_corresponde_ao_mercado": "salario_atual_n_corresponde_mercado",
    "Nao_tenho_uma_boa_relacao_com_meu_lider_gestor": "rel_ruim_lider_gestor",
    "Gostaria_de_trabalhar_em_em_outra_area_de_atuacao": "trab_outra_area_atuacao",
    "Gostaria_de_receber_mais_beneficios": "mais_beneficios",
    "O_clima_de_trabalho_ambiente_nao_e_bom": "clima_trabalho_ruim",
    "Falta_de_maturidade_analitica_na_empresa": "falta_maturidade_analitica_empresa",
    "Voce_participou_de_entrevistas_de_emprego_nos_ultimos_6_meses": "participou_entrevistas_ult_6_meses",
    "Voce_pretende_mudar_de_emprego_nos_proximos_6_meses": "mudar_emprego_prox_6_meses",
    "Quais_os_principais_criterios_que_voce_leva_em_consideracao_no_momento_de_decidir_onde_trabalhar": "principais_criterios_levados_decidir_trab",
    "Remuneracao_Salario": "remuneracao_salario",
    "Beneficios": "beneficios",
    "Proposito_do_trabalho_e_da_empresa": "proposito_trabalho_empresa",
    "Flexibilidade_de_trabalho_remoto": "flexibilidade_trab_remoto",
    "Ambiente_e_clima_de_trabalho": "ambiente_clima_trabalho",
    "Oportunidade_de_aprendizado_e_trabalhar_com_referencias_na_area": "oport_aprendizado_trab_ref_area",
    "Plano_de_carreira_e_oportunidades_de_crescimento_profissional": "plano_carreira_oport_cresc_prof",
    "Maturidade_da_empresa_em_termos_de_tecnologia_e_dados": "maturidade_empresa_dados_tec",
    "Qualidade_dos_gestores_e_lideres": "qualidade_lideres_gestores",
    "Reputacao_que_a_empresa_tem_no_mercado": "rep_empresa_mercado",
    "Empresa_que_trabaha_passou_por_layoff_em_2023": "empresa_trabalha_passou_layoff_2023",
    "Atualmente_qual_a_sua_forma_de_trabalho": "forma_trabalho",
    "Qual_a_forma_de_trabalho_ideal_para_voce": "forma_trabalho_ideal",
    "Caso_sua_empresa_decida_pelo_modelo_100_presencial_qual_sera_sua_atitude": "atitude_perante_100_presencial",
    "Qual_o_numero_aproximado_de_pessoas_que_atuam_com_dados_na_sua_empresa_hoje": "num_prof_dados_atuando_empresa_hoje",
    "Quais_desses_papeis_cargos_fazem_parte_do_time_ou_chapter_de_dados_da_sua_empresa": "papeis_cargos_no_time_empresa",
    "Analytics_Engineer": "engenheiro_analytics",
    "Engenharia_de_Dados_Data_Engineer": "engenheiro_dados",
    "Analista_de_Dados_Data_Analyst": "analista_dados",
    "Cientista_de_Dados_Data_Scientist": "cientista_dados",
    "Database_Administrator_DBA": "dba",
    "Analista_de_Business_Intelligence_BI": "analista_bi",
    "Arquiteto_de_Dados_Data_Architect": "arquiteto_dados",
    "Data_Product_Manager_DPM": "data_product_manager",
    "Business_Analyst": "analista_business",

    "Quais_dessas_responsabilidades_fazem_parte_da_sua_rotina_atual_de_trabalho_como_gestor": "resp_rotina_atual_como_gestor",
    "Pensar_na_visao_de_longo_prazo_de_dados_da_empresa_e_fortalecimento_da_cultura_analitica_da_companhia": "visao_longo_prazo_dados_empresa",
    "Organizacao_de_treinamentos_e_iniciativas_com_o_objetivo_de_aumentar_a_maturidade_analitica_das_areas_de_negocios": "organizacao_treinamentos_iniciat_para_maior_maturidade_analitica",
    "Atracao_selecao_e_contratacao_de_talentos_para_o_time_de_dados": "contratacao_area_dados",
    "Decisao_sobre_contratacao_de_ferramentas_e_tecnologias_relacionadas_a_dados": "contratacao_ferramentas_dados",
    "Sou_gestor_da_equipe_responsavel_pela_engenharia_de_dados_e_por_manter_o_Data_Lake_da_empresa_como_fonte_unica_dos_dados_garantindo_a_qualidade_e_confiabilidade_da_informacao": "gestor_eng_dados",
    "Sou_gestor_da_equipe_responsavel_pela_entrega_de_dados_estudos_relatorios_e_dashboards_para_as_areas_de_negocio_da_empresa": "gestor_analise_dados",
    "Sou_gestor_da_equipe_responsavel_por_iniciativas_e_projetos_envolvendo_Inteligencia_Artificial_e_Machine_Learning": "gestor_ia",
    "Apesar_de_ser_gestor_ainda_atuo_na_parte_tecnica_construindo_solucoes_analises_modelos_etc": "gestor_ciencia_dados",
    "Gestao_de_projetos_de_dados_cuidando_das_etapas_equipes_envolvidas_atingimento_dos_objetivos_etc": "gestor_projetos",
    "Gestao_de_produtos_de_dados_cuidando_da_visao_dos_produtos_backlog_feedback_de_usuarios_etc": "gestor_produtos",
    "Gestao_de_pessoas_apoio_no_desenvolvimento_das_pessoas_evolucao_de_carreira": "gestor_pessoas",
    "Quais_sao_os_3_maiores_desafios_que_voce_tem_como_gestor_no_atual_momento": "tres_maiores_desafios",
    "a_Contratar_novos_talentos": "contratar_novos_talentos",
    "b_Reter_talentos": "reter_talentos",
    "c_Convencer_a_empresa_a_aumentar_os_investimentos_na_area_de_dados": "convencer_aumentar_invest_dados",
    "d_Gestao_de_equipes_no_ambiente_remoto": "gestao_equipe_em_remoto",
    "e_Gestao_de_projetos_envolvendo_areas_multidisciplinares_da_empresa": "gestao_proj_multidisciplinaridades_empresa",
    "f_Organizar_as_informacoes_e_garantir_a_qualidade_e_confiabilidade": "organizar_info_resguard_quali_confiab",
    "g_Conseguir_processar_e_armazenar_um_alto_volume_de_dados": "processar_armazenar_big_data",
    "h_Conseguir_gerar_valor_para_as_areas_de_negocios_atraves_de_estudos_e_experimentos": "gerar_valor_area_neg_atrav_estud_experim",
    "i_Desenvolver_e_manter_modelos_Machine_Learning_em_producao": "desenv_e_mant_modelos_ml_prod",
    "j_Gerenciar_a_expectativa_das_areas_de_negocio_em_relacao_as_entregas_das_equipes_de_dados": "gerenciar_expectativ_areas_neg_rel_equipe_dados",
    "k_Garantir_a_manutencao_dos_projetos_e_modelos_em_producao_em_meio_ao_crescimento_da_empresa": "manutencao_projetos_model_prod_meio_cresc_empresa",
    "Conseguir_levar_inovacao_para_a_empresa_atraves_dos_dados": "inovacao_atraves_area_dados",
    "Garantir_retorno_do_investimento_ROI_em_projetos_de_dados": "retorno_roi_projeto_dados",
    "Dividir_o_tempo_entre_entregas_tecnicas_e_gestao": "dividir_tempo_entregas_tec_gestao",
    "AI_Generativa_e_uma_prioridade_em_sua_empresa": "ia_gen_eh_prioridade_empresa",
    "Tipos_de_uso_de_AI_Generativa_e_LLMs_na_empresa": "tipo_uso_ia_generativa_llm_empresa",
    "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada": "colaboradores_uso_ia_descentra_independ",
    "Direcionamento_centralizado_do_uso_de_AI_generativa": "direcionamento_centralizado_ia_generativa",
    "Desenvolvedores_utilizando_Copilots": "dev_usando_copilot",
    "AI_Generativa_e_LLMs_para_melhorar_produtos_externos": "ia_gen_llm_melhorar_prod_ext",
    "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores": "ia_gen_llm_melhorar_prod_int_colaboradores",
    "IA_Generativa_e_LLMs_como_principal_frente_do_negocio": "ia_gen_llm_principal_frente_neg",
    "IA_Generativa_e_LLMs_nao_e_prioridade": "ia_llm_nao_prioridade",
    "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa": "sem_opiniao_llm_ia_generativa",
    "Motivos_que_levam_a_empresa_a_nao_usar_AI_Genrativa_e_LLMs": "motiv_empresa_nao_usar_ia_gen_llm",
    "Falta_de_compreensao_dos_casos_de_uso": "falta_compreensao_caso_uso",
    "Falta_de_confiabilidade_das_saidas_alucinacao_dos_modelos": "falta_confiab_saida_alucinacao_modelo",
    "Incerteza_em_relacao_a_regulamentacao": "incerteza_rel_regulamentacao",
    "Preocupacoes_com_seguranca_e_privacidade_de_dados": "seguranca_privacidade_dados",
    "Retorno_sobre_investimento_ROI_nao_comprovado_de_IA_Generativa": "retorno_roi_nao_comprovado_ia_gen",
    "Dados_da_empresa_nao_estao_prontos_para_uso_de_IA_Generativa": "dados_empresa_nao_prep_ia_gen",
    "Falta_de_expertise_ou_falta_de_recursos": "falta_expertise_ou_recurso",
    "Alta_direcao_da_empresa_nao_ve_valor_ou_nao_ve_como_prioridade": "alta_direcao_nao_ve_valor_prioridade",
    "Preocupacoes_com_propriedade_intelectual": "preocupacao_prop_intelectual",
    "Mesmo_que_esse_nao_seja_seu_cargo_formal_voce_considera_que_sua_atuacao_no_dia_a_dia_reflete_alguma_das_opcoes_listadas_abaixo": "nao_sendo_cargo_exerce_funcao",
    "Atuacao": "atuacao",
    "Quais_das_fontes_de_dados_listadas_voce_ja_analisou_ou_processou_no_trabalho": "fontes_dados_analisou_trab",
    "Dados_relacionais_estruturados_em_bancos_SQL": "banco_relacional",
    "Dados_armazenados_em_bancos_NoSQL": "banco_no_sql",
    "Imagens": "img",
    "Textos_Documentos": "text_doc",
    "Videos": "video",
    "Audios": "audio",
    "Planilhas": "planilha",
    "Dados_georeferenciados": "georefenciados",
    "Entre_as_fontes_de_dados_listadas_quais_voce_utiliza_na_maior_parte_do_tempo": "fonte_dados_mais_utilizadas",
    "Dados_relacionais_estruturados_em_bancos_SQL_1": "banco_relacional_fonte",
    "Dados_armazenados_em_bancos_NoSQL_1": "banco_no_sql_fonte",
    "Imagens_1": "img_fonte",
    "Textos_Documentos_1": "text_doc_fonte",
    "Videos_1": "video_fonte",
    "Audios_1": "audio_fonte",
    "Planilhas_1": "planilha_fonte",
    "Dados_georeferenciados_1": "georeferenciados_fonte",
    "Quais_das_linguagens_listadas_abaixo_voce_utiliza_no_trabalho": "linguagens_utilizadas_trab",
    "SQL": "sql",
    "R": "r",
    "Python": "python",
    "C_C_C": "ccc",
    "NET": "net",
    "Java": "java",
    "Julia": "julia",
    "SAS_Stata": "sas_stat",
    "Visual_Basic_VBA": "vba",
    "Scala": "scala",
    "Matlab": "matlab",
    "Rust": "rust",
    "PHP": "php",
    "JavaScript": "js",
    "Nao_utilizo_nenhuma_linguagem": "nao_utilizo_nenhuma",
    "Entre_as_linguagens_listadas_abaixo_qual_e_a_que_voce_mais_utiliza_no_trabalho": "linguagem_mais_utilizada_trab",
    "Entre_as_linguagens_listadas_abaixo_qual_e_a_sua_preferida": "linguagem_preferida",
    "Quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho": "banco_relacional_utilizado",
    "MySQL": "my_sql",
    "Oracle": "oracle",
    "SQL_SERVER": "sql_server",
    "Amazon_Aurora_ou_RDS": "amazon_aurora_rds",
    "DynamoDB": "dynamo_db",
    "CoachDB": "coach_db",
    "Cassandra": "cassandra",
    "MongoDB": "mongo_db",
    "MariaDB": "maria_db",
    "Datomic": "datomic",
    "S3": "s3",
    "PostgreSQL": "postgre_sql",
    "ElasticSearch": "elastic_search",
    "DB2": "db2",
    "Microsoft_Access": "microsoft_acess",
    "SQLite": "sqlite",
    "Sybase": "sybase",
    "Firebase": "firebase",
    "Vertica": "vertica",
    "Redis": "redis",
    "Neo4J": "neo4j",
    "Google_BigQuery": "google_big_query",
    "Google_Firestore": "google_firestore",
    "Amazon_Redshift": "amazon_redshift",
    "Amazon_Athena": "amazon_athena",
    "Snowflake": "snowflake",
    "Databricks": "databricks",
    "HBase": "hbase",
    "Presto": "presto",
    "Splunk": "splunk",
    "SAP_HANA": "sap_hana",
    "Hive": "hive",
    "Firebird": "firebird",
    "Dentre_as_opcoes_listadas_qual_sua_Cloud_preferida": "cloud_preferida",
    "Azure_Microsoft": "azure",
    "Amazon_Web_Services_AWS": "aws",
    "Google_Cloud_GCP": "gcp",
    "Oracle_Cloud": "oracle",
    "IBM": "ibm",
    "Servidores_On_Premise_Nao_utilizamos_Cloud": "on_premise",
    "Cloud_Propria": "cloud_proprietaria",
    "Cloud_preferida": "cloud_preferida",
    "Ferramenta_de_BI_utilizada_no_dia_a_dia": "ferramenta_bi_diaria",
    "Microsoft_PowerBI": "pbi",
    "Qlik_View_Qlik_Sense": "qlik_view_sense",
    "Tableau": "tableau",
    "Metabase": "metabase",
    "Superset": "superset",
    "Redash": "redash",
    "Looker": "looker",
    "Looker_Studio_Google_Data_Studio": "looker_studio",
    "Amazon_Quicksight": "amazon_quicksight",
    "Mode": "mode",
    "Alteryx": "alteryx",
    "MicroStrategy": "microstategy",
    "IBM_Analytics_Cognos": "ibm_analytics_cognos",
    "SAP_Business_Objects_SAP_Analytics": "sap_business_objects",
    "Oracle_Business_Intelligence": "oracle_bi",
    "Salesforce_Einstein_Analytics": "salesforce_einstein_analytics",
    "Birst": "birst",
    "SAS_Visual_Analytics": "sas_visual_analytics",
    "Grafana": "grafana",
    "TIBCO_Spotfire": "tibco_spotfire",
    "Pentaho": "pentaho",
    "Fazemos_todas_as_analises_utilizando_apenas_Excel_ou_planilhas_do_google": "excel_planilha_apenas",
    "Nao_utilizo_nenhuma_ferramenta_de_BI_no_trabalho": "nenhuma_ferramenta_bi",
    "Qual_sua_ferramenta_de_BI_preferida": "ferramenta_bi_favorita",
    "Qual_o_tipo_de_uso_de_AI_Generativa_e_LLMs_na_empresa": "tipo_uso_ia_llm_empresa",
    "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_1": "ia_llm_empresa_descentralizada_independente",
    "Direcionamento_centralizado_do_uso_de_AI_generativa_1": "ia_llm_empresa_uso_centralizado",
    "Desenvolvedores_utilizando_Copilots_1": "ia_llm_empresa_uso_copilot",
    "AI_Generativa_e_LLMs_para_melhorar_produtos_externos_para_os_clientes_finais": "ia_llm_empresa_prod_externo",
    "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_1": "ia_llm_empresa_prod_interno",
    "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_1": "ia_llm_empresa_frente_negoc",
    "IA_Generativa_e_LLMs_nao_e_prioridade_1": "ia_llm_empresa_nao_prioridade",
    "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_1": "ia_llm_empresa_sem_opiniao",
    "Utiliza_ChatGPT_ou_LLMs_no_trabalho": "utilizacao_chat_gpt_llm",
    "Nao_uso_solucoes_de_AI_Generativa_com_foco_em_produtividade": "nao_uso_ia_gen_produtividade",
    "Uso_solucoes_gratuitas_de_AI_Generativa_com_foco_em_produtividade": "uso_ia_gen_gratuita_produtividade",
    "Uso_e_pago_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade": "uso_pago_ia_gen_produtividade",
    "A_empresa_que_trabalho_paga_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade": "uso_pago_ia_gen_empresa_paga",
    "Uso_solucoes_do_tipo_Copilot": "uso_copilot",
    "Qual_seu_objetivo_na_area_de_dados": "obj_area_dados",
    "Qual_oportunidade_voce_esta_buscando": "oportunidade_almejada",
    "Ha_quanto_tempo_voce_busca_uma_oportunidade_na_area_de_dados": "tempo_busca_oport_area_dados",
    "Como_tem_sido_a_busca_por_um_emprego_na_area_de_dados": "busca_empreg_area_dados",
    "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_como_engenheiro_de_dados": "rotina_engenheiro_dados",
    "Desenvolvo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc": "pipeline_dados_ling_prog",
    "Realizo_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc": "etl_pentaho_talent_etc",
    "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio": "relatorios_sql_exportados",
    "Atuo_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc": "integracao_dados_plataformas_dados",
    "Modelo_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao": "criacao_componentes_ingestao_dados",
    "Desenvolvo_cuido_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses": "manutencao_criacao_rep_dados_streaming",
    "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc": "modelagem_dados_dw_data_mart",
    "Cuido_da_qualidade_dos_dados_metadados_e_dicionario_de_dados": "quali_dados_dicionario",
    "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia": "nenhuma_opcao_engenheiro_dados",
    "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Engineer": "ferramenta_etl_utilizado_engenheiro_dados",
    "Scripts_Python": "script_python",
    "SQL_Stored_Procedures": "sql_stored_procedures",
    "Apache_Airflow": "apache_airflow",
    "Apache_NiFi": "apache_nifi",
    "Luigi": "luigi",
    "AWS_Glue": "aws_glue",
    "Talend": "talend",
    "Pentaho_1": "engenheiro_dados_pentaho",
    "Alteryx_1": "engenheiro_dados_alteryx",
    "Stitch": "stitch",
    "Fivetran": "fivetran",
    "Google_Dataflow": "google_dataflow",
    "Oracle_Data_Integrator": "oracle_data_integrator",
    "IBM_DataStage": "ibm_data_stage",
    "SAP_BW_ETL": "sap_bw_etl",
    "SQL_Server_Integration_Services_SSIS": "SSIS",
    "SAS_Data_Integration": "sas_data_integration",
    "Qlik_Sense": "qlik_sense",
    "Knime": "knime",
    "Databricks_1": "engenheiro_dados_databricks",
    "Nao_utilizo_ferramentas_de_ETL": "engenheiro_dados_sem_ferramenta_etl",
    "Sua_organizacao_possui_um_Data_Lake": "empresa_possui_datalake",
    "Qual_tecnologia_utilizada_como_plataforma_do_Data_Lake": "tecnologia_utilizada_datalake",
    "Sua_organizacao_possui_um_Data_Warehouse": "empresa_possui_dw",
    "Qual_tecnologia_utilizada_como_plataforma_do_Data_Warehouse": "tecnologia_utilizada_dw",
    "Quais_as_ferramentas_de_gestao_de_Qualidade_de_dados_Metadados_e_catalogo_de_dados_voce_utiliza_no_trabalho": "ferramentas_meta_dados_qualidade_dados_trabalho",
    "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo": "maior_parte_tempo_gasta",
    "Desenvolvendo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc": "pipelines_com_codigo",
    "Realizando_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc": "etl_com_ferramentas_bi",
    "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio": "exportacao_rel_para_area_neg",
    "Atuando_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc": "integracao_fonte_dados_diferentes_tmp_gasto",
    "Modelando_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao": "modelagem_arquitetura_dados_tmp_gasto",
    "Desenvolvendo_cuidando_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses": "dados_streaming_data_lake_lakehouse",
    "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc": "modelagem_dados_dw_data_mart",
    "Cuidando_da_qualidade_dos_dados_metadados_e_dicionario_de_dados": "qualidade_dados_tmp_gasto",
    "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_1": "nenhuma_das_opcoes_tmp_gasto",
    "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_analise_de_dados": "analise_dados_rotina",
    "Processo_e_analiso_dados_utilizando_linguagens_de_programacao_como_Python_R_etc": "analise_processamento_linguagem_prog",
    "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc": "dashboard_ferramentas_bi",
    "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_1": "exportacao_rel_area_neg_analista_dados",
    "Utilizo_API_s_para_extrair_dados_e_complementar_minhas_analises": "extracao_dados_api",
    "Realizo_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc": "estudo_experimentos_util_estatistica",
    "Desenvolvo_cuido_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc": "manutencao_etl_utilizando_ferramenta_etl_analista",
    "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc": "modelagem_dw_data_mart_analista",
    "Desenvolvo_cuido_da_manutencao_de_planilhas_para_atender_as_areas_de_negocio": "manutencao_planilha_area_negocio",
    "Utilizo_ferramentas_avancadas_de_estatistica_como_SASS_PSS_Stata_etc": "utilizacao_ferramenta_avancada_estatistica",
    "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_2": "nenhuma_das_opcoes_analista",
    "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Analyst": "ferramental_analista_dados",
    "Scripts_Python_1": "python_analista",
    "SQL_Stored_Procedures_1": "stored_procedures_analista",
    "Apache_Airflow_1": "apache_airflow_analista",
    "Apache_NiFi_1": "apache_nifi_analista",
    "Luigi_1": "luigi_analista",
    "AWS_Glue_1": "aws_glue_analista",
    "Talend_1": "talent_analista",
    "Pentaho_2": "pentaho_analista",
    "Alteryx_2": "alteryx_analista",
    "Stitch_1": "stitch_analista",
    "Fivetran_1": "fivetran_analista",
    "Google_Dataflow_1": "google_dataflow_analista",
    "Oracle_Data_Integrator_1": "oracle_data_integrator_analista",
    "IBM_DataStage_1": "ibm_data_stage_analista",
    "SAP_BW_ETL_1": "sap_bw_etl_analista",
    "SQL_Server_Integration_Services_SSIS_1": "ssis_analista",
    "SAS_Data_Integration_1": "sas_data_integration_analista",
    "Qlik_Sense_1": "qlik_sense_analista",
    "Knime_1": "knime_analista",
    "Databricks_2": "databricks_analista",
    "Nao_utilizo_ferramentas_de_ETL_1": "nao_utilizo_ferramentas_etl_analista",
    "Sua_empresa_utiliza_alguma_das_ferramentas_listadas_para_dar_mais_autonomia_em_analise_de_dados_para_as_areas_de_negocio": "ferramentas_empresa_usadas_mais_autonomia_area_neg",
    "Ferramentas_de_AutoML_como_H2O_ai_Data_Robot_BigML_etc": "ferramentas_auto_ml",
    "Point_and_Click_Analytics_como_Alteryx_Knime_Rapidminer_etc": "point_click",
    "Product_metricts_Insights_como_Mixpanel_Amplitude_Adobe_Analytics": "product_metrics_insight",
    "Ferramentas_de_analise_dentro_de_ferramentas_de_CRM_como_Salesforce_Einstein_Anaytics_ou_Zendesk_dashboards": "ferramentas_analise_dados_dentro_crm",
    "Minha_empresa_nao_utiliza_essas_ferramentas": "empresa_nao_utiliza_ferramenta_autonomia",
    "Nao_sei_informar": "nao_sei_informar_ferramenta_autonomia_area_neg",
    "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_de_trabalho": "maior_parte_tempo_gasto_analista",
    "Processando_e_analisando_dados_utilizando_linguagens_de_programacao_como_Python_R_etc": "dashboards_script_analista",
    "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc": "dashboards_ferramenta_bi_analista",
    "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_1": "extracao_rel_sql_analista",
    "Utilizando_API_s_para_extrair_dados_e_complementar_minhas_analises": "extracao_api_analista",
    "Realizando_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc": "estudo_uso_estatistica_analista",
    "Desenvolvendo_cuidando_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc": "criacao_manutencao_etl_analista",
    "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc": "modelagem_dados_dw_data_mart_analista",
    "Desenvolvendo_cuidando_da_manutencao_de_planilhas_do_Excel_ou_Google_Sheets_para_atender_as_areas_de_negocio": "manutencao_planilhas_analista",
    "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises": "ferramentas_avanc_estatistica_analista",
    "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_3": "nenhuma_opcao_listada_atv_analista",
    "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_ciencia_de_dados": "rotina_trabalho_cientista",
    "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio": "uso_adhoc_modelo_preditivo",
    "Sou_responsavel_pela_coleta_e_limpeza_dos_dados_que_uso_para_analise_e_modelagem": "coleta_limpeza_modelagem_analise",
    "Sou_responsavel_por_entrar_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados": "analise_requisitos_area_negocio",
    "Desenvolvo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados": "dev_modelo_ml",
    "Sou_responsavel_por_colocar_modelos_em_producao_criar_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento": "deploy_modelo_prod_criacao_api_monitoramento",
    "Cuido_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario": "manutencao_modelo_ml_producao_melhorias",
    "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_1": "construcao_dashboard_ferramenta_bi_analista",
    "Utilizo_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_estatisticas_e_ajustar_modelos": "ferramenta_avanc_estatistica_analise_ajust_modelo",
    "Crio_e_dou_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados": "criacao_manun_etl_dags_automacoes",
    "Crio_e_gerencio_solucoes_de_Feature_Store_e_cultura_de_MLOps": "criacao_gerencia_feature_store_mlops",
    "Sou_responsavel_por_criar_e_manter_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc": "infra_modelos_rodam_em_clusters_api_etc",
    "Treino_e_aplico_LLM_s_para_solucionar_problemas_de_negocio": "criacao_treinamento_llm",
    "Quais_as_tecnicas_e_metodos_listados_abaixo_voce_costuma_utilizar_no_trabalho": "tecnica_metodo_utilizado_cientista_trabalho",
    "Utilizo_modelos_de_regressao_linear_logistica_GLM": "regressao_logistica_glm",
    "Utilizo_redes_neurais_ou_modelos_baseados_em_arvore_para_criar_modelos_de_classificacao": "redes_neurais_odelo_arvore",
    "Desenvolvo_sistemas_de_recomendacao_RecSys": "sistema_recomendacao",
    "Utilizo_metodos_estatisticos_Bayesianos_para_analisar_dados": "metodo_estatistico_bayesiano_analise",
    "Utilizo_tecnicas_de_NLP_Natural_Language_Processing_para_analisar_dados_nao_estruturados": "NLP_dados_nao_estruturados",
    "Utilizo_metodos_estatisticos_classicos_Testes_de_hipotese_analise_multivariada_sobrevivencia_dados_longitudinais_inferencia_estatistica_para_analisar_dados": "metodos_estatisticos_classicos",
    "Utilizo_cadeias_de_Markov_ou_HMM_s_para_realizar_analises_de_dados": "cadeias_markov_hmm_analise_dados",
    "Desenvolvo_tecnicas_de_Clusterizacao_K_means_Spectral_DBScan_etc": "tecnicas_clusterizacao",
    "Realizo_previsoes_atraves_de_modelos_de_Series_Temporais_Time_Series": "modelos_preditivos",
    "Utilizo_modelos_de_Reinforcement_Learning_aprendizado_por_reforco": "aprendizado_reforco",
    "Utilizo_modelos_de_Machine_Learning_para_deteccao_de_fraude": "ml_deteccao_fraude",
    "Utilizo_metodos_de_Visao_Computacional": "visao_computacional",
    "Utilizo_modelos_de_Deteccao_de_Churn": "modelo_deteccao_churn",
    "Utilizo_LLM_s_para_solucionar_problemas_de_negocio": "llm_cientista_dados_uso",
    "Quais_dessas_tecnologias_fazem_parte_do_seu_dia_a_dia_como_cientista_de_dados": "tecnologias_diarias_cientista",
    "Ferramentas_de_BI_PowerBI_Looker_Tableau_Qlik_etc": "ferramenta_bi_cientista",
    "Planilhas_Excel_Google_Sheets_etc": "planilha_excel_cientista",
    "Ambientes_de_desenvolvimento_local_R_studio_JupyterLab_Anaconda": "ambiente_local_jupyter_anaconda",
    "Ambientes_de_desenvolvimento_na_nuvem_Google_Colab_AWS_Sagemaker_Kaggle_Notebooks_etc": "dev_nuvem",
    "Ferramentas_de_AutoML_Datarobot_H2O_Auto_Keras_etc": "auto_ml_cientista",
    "Ferramentas_de_ETL_Apache_Airflow_NiFi_Stitch_Fivetran_Pentaho_etc": "ferramenta_etl_cientista",
    "Plataformas_de_Machine_Learning_TensorFlow_Azure_Machine_Learning_Kubeflow_etc": "plataforma_ml_cientista",
    "Feature_Store_Feast_Hopsworks_AWS_Feature_Store_Databricks_Feature_Store_etc": "feature_store_cientista",
    "Sistemas_de_controle_de_versao_Github_DVC_Neptune_Gitlab_etc": "controle_de_versao_cientista",
    "Plataformas_de_Data_Apps_Streamlit_Shiny_Plotly_Dash_etc": "data_apps_cientista",
    "Ferramentas_de_estatistica_avancada_como_SPSS_SAS_etc": "ferramenta_estatistica_avanc_cientista",
    "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_no_trabalho": "tempo_gasto_cientista",
    "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_1": "ad_hoc_hipoteses_tmp_gasto_cientista",
    "Coletando_e_limpando_os_dados_que_uso_para_analise_e_modelagem": "coleta_limpeza_cientista",
    "Entrando_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados": "analise_requisitos_tmp_gasto_cientista",
    "Desenvolvendo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados": "dev_ml_prod_tmp_gasto_cientista",
    "Colocando_modelos_em_producao_criando_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento": "manutencao_ml_prod_tmp_gasto_cientista",
    "Cuidando_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario": "manutencao_ml_em_prod_monitoramento_ajuste_tmp_gasto_cientista",
    "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_1": "dashboards_ferramenta_bi_tmp_gasto_cientista",
    "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_1": "ferramenta_avancadas_estatistica_analista_tmp_gasto_cientista",
    "Criando_e_dando_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados": "manutencao_etl_dag_tmp_gasto_cientista",
    "Criando_e_gerenciando_solucoes_de_Feature_Store_e_cultura_de_MLOps": "feature_store_ml_ops_tmp_gasto_cientista",
    "Criando_e_mantendo_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc": "infra_modelo_clusters_servidores_tmp_gasto_cientista",
    "Treinando_e_aplicando_LLM_s_para_solucionar_problemas_de_negocio": "LLM_tmp_gasto_cientista"
}

In [5]:
# caminho_2023 = "s3://s3-state-data/state_data_2023.csv"
# caminho_2024 = "s3://s3-state-data/state_data_2024.csv"
# caminho_2025 = "s3://s3-state-data/state_data_2025.csv"


caminho_2023 = "/content/state_data_2023.csv"
caminho_2024 = "/content/state_data_2024.csv"
caminho_2025 = "/content/state_data_2025.csv"

# Criação da camada bronze do DW da pesquisa para 2023

In [6]:
df_state_data_2023 = ler_df_csv(sessao, caminho_2023)
df_state_data_2023 = renomear_colunas(df_state_data_2023, novos_nomes_colunas_2023)
df_state_data_2023 = processar_dataframe(df_state_data_2023)

In [7]:
df_state_data_2023.show()

+--------------------+-----+------------+---------+--------------------+---+--------------------+-------------------+------------------------------+-------------------------+-------------------+--------------------+-------------------------------------+-------------------------------------------+-------------------------------------+--------------------------------+-----------------------+--------------------------+----------------------+--------------------------------------+-----------------------------------------------+-----------+--------------------+---+------------+------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------+------+--------------------+--------------------+------+--------------------+---------------+--------------------+------------------------+----------------------------+------------------------+-----------------------------------+---------------------+-----------------------+--------------

In [8]:
exportar_df_para_csv(df_state_data_2023, "bronze_dw_state_data_2023.csv")

# Criação camada bronze do DW para a pesquisa de 2024

In [9]:
novos_nomes_colunas_2024 = {
    "0.a_token": "token_user",
    "0.d_data/hora_envio": "data_hora_envio",
    "1.a_idade": "idade",
    "1.a.1_faixa_idade": "faixa_etaria",
    "1.b_genero": "genero",
    "1.c_cor/raca/etnia": "cor_raca_etnia",
    "1.d_pcd": "pcd",
    "1.e_experiencia_profissional_prejudicada": "exp_prof_prejud",
    "1.e.1_Não acredito que minha experiência profissional seja afetada": "exp_prof_nao_prejud",
    "1.e.2_Sim, devido a minha Cor/Raça/Etnia": "exp_prof_prejud_cor_raca_etnia",
    "1.e.3_Sim, devido a minha identidade de gênero": "exp_prof_prejud_ident_gen",
    "1.e.4_Sim, devido ao fato de ser PCD": "exp_prof_prejud_pcd",
    "1.i.1_uf_onde_mora": "uf",
    "1.f.1_Quantidade de oportunidades de emprego/vagas recebidas": "qtd_oportunidades_vagas_emprego_receb",
    "1.f.2_Senioridade das vagas recebidas em relação à sua experiência": "senioridade_vagas_recebidas_rel_experiencia",
    "1.f.3_Aprovação em processos seletivos/entrevistas": "aprov_processos_seletivos_entrevistas",
    "1.f.4_Oportunidades de progressão de carreira": "oportunidades_progresso_carreira",
    "1.f.5_Velocidade de progressão de carreira": "vel_progressao_carreira",
    "1.f.6_Nível de cobrança no trabalho/Stress no trabalho": "nvl_cobranca_e_stress_trab",
    "1.f.7_Atenção dada pelas pessoas diante das minhas opiniões e ideias": "atencao_opiniao_ideias",
    "1.f.8_Relação com outras pessoas da empresa, em momentos de trabalho": "rel_outros_membros_empresa_em_trabalho",
    "1.f.9_Relação com outras pessoas da empresa, em momentos de integração e outros momentos fora do trabalho": "rel_outros_membros_empresa_integracao_fora_trab",
    "1.i.2_regiao_onde_mora": "regiao",
    "1.f_aspectos_prejudicados": "aspectos_prejud",
    "1.k.1_uf_de_origem": "uf",
    "1.k.2_regiao_de_origem": "regiao_origem",
    "1.g_vive_no_brasil": "vive_brasil",
    "1.h_pais_onde_mora": "pais",
    "1.i_estado_onde_mora": "estado",
    "1.j_vive_no_estado_de_formacao": "vive_estado_formacao",
    "1.k_estado_de_origem": "estado_origem",
    "1.l_nivel_de_ensino": "nvl_ensino",
    "1.m_área_de_formação": "area_formacao",

    "2.a_situação_de_trabalho": "sit_atual_trab",
    "2.b_setor": "setor",
    "2.c_numero_de_funcionarios": "n_funcionarios",
    "2.d_atua_como_gestor": "gestor",
    "2.e_cargo_como_gestor": "cargo_gestor",
    "2.f_cargo_atual": "cargo_atual",
    "2.g_nivel": "nvl",
    "2.h_faixa_salarial": "faixa_salarial",
    "2.i_tempo_de_experiencia_em_dados": "tempo_exp_dados",
    "2.j_tempo_de_experiencia_em_ti": "tempo_exp_ti",
    "2.k_satisfeito_atualmente": "satisfacao_empresa_atual",
    "2.l.1_Remuneração/Salário": "remuneracao_salario",
    "2.l.2_Benefícios": "beneficios",
    "2.l.3_Propósito do trabalho e da empresa": "proposito_trabalho_empresa",
    "2.l.4_Flexibilidade de trabalho remoto": "flexibilidade_trab_remoto",
    "2.l.5_Ambiente e clima de trabalho": "ambiente_clima_trabalho",
    "2.l.6_Oportunidade de aprendizado e trabalhar com referências": "oport_aprendizado_trab_ref_area",
    "2.l.7_Oportunidades de crescimento": "plano_carreira_oport_cresc_prof",
    "2.l.8_Maturidade da empresa em termos de tecnologia e dados": "maturidade_empresa_dados_tec",
    "2.l.9_Relação com os gestores e líderes": "qualidade_lideres_gestores",
    "2.l.10_Reputação que a empresa tem no mercado": "rep_empresa_mercado",
    "2.l.11_Gostaria de trabalhar em outra área": "trab_outra_area_atuacao",
    "2.l_motivo_insatisfacao": "motivo_insatis_empresa_atual",
    "2.m_participou_de_entrevistas_ultimos_6m": "participou_entrevistas_ult_6_meses",
    "2.n_planos_de_mudar_de_emprego_6m": "mudar_emprego_prox_6_meses",
    "2.o_criterios_para_escolha_de_emprego": "principais_criterios_levados_decidir_trab",
    "2.o.1_Remuneração/Salário": "remuneracao_salario",
    "2.o.2_Benefícios": "beneficios",
    "2.o.3_Propósito do trabalho e da empresa": "proposito_trabalho_empresa",
    "2.o.4_Flexibilidade de trabalho remoto": "flexibilidade_trab_remoto",
    "2.o.5_Ambiente e clima de trabalho": "ambiente_clima_trabalho",
    "2.o.6_Oportunidade de aprendizado e trabalhar com referências": "oport_aprendizado_trab_ref_area",
    "2.o.7_Plano de carreira e oportunidades de crescimento": "plano_carreira_oport_cresc_prof",
    "2.o.8_Maturidade da empresa em termos de tecnologia e dados": "maturidade_empresa_dados_tec",
    "2.o.9_Qualidade dos gestores e líderes": "qualidade_lideres_gestores",
    "2.o.10_Reputação que a empresa tem no mercado": "rep_empresa_mercado",
    "2.q_empresa_passou_por_layoff_em_2024": "layoff_2024",
    "2.r_modelo_de_trabalho_atual": "forma_trabalho",
    "2.s_modelo_de_trabalho_ideal": "forma_trabalho_ideal",
    "2.t_atitude_em_caso_de_retorno_presencial": "atitude_perante_100_presencial",

    "3.a_numero_de_pessoas_em_dados": "num_prof_dados_atuando_empresa_hoje",
    "3.b_cargos_no_time_de_dados_da_empresa": "papeis_cargos_no_time_empresa",
    "3.b.1_Analytics Engineer": "engenheiro_analytics",
    "3.b.2_Engenharia de Dados/Data Engineer": "engenheiro_dados",
    "3.b.3_Analista de Dados/Data Analyst": "analista_dados",
    "3.b.4_Cientista de Dados/Data Scientist": "cientista_dados",
    "3.b.5_Database Administrator/DBA": "dba",
    "3.b.6_Analista de Business Intelligence/BI": "analista_bi",
    "3.b.7_Arquiteto de Dados/Data Architect": "arquiteto_dados",
    "3.b.8_Data Product Manager/DPM": "data_product_manager",
    "3.b.9_Business Analyst": "analista_business",
    "3.b.10_ML Engineer/AI Engineer": "engenheiro_ml_ia",

    "3.c_responsabilidades_como_gestor": "resp_rotina_atual_como_gestor",
    "3.c.1_Pensar na visão de longo prazo de dados": "visao_longo_prazo_dados_empresa",
    "3.c.2_Organização de treinamentos e iniciativas": "organizacao_treinamentos_iniciat_para_maior_maturidade_analitica",
    "3.c.3_Atração, seleção e contratação": "contratacao_area_dados",
    "3.c.4_Decisão sobre contratação de ferramentas": "contratacao_ferramentas_dados",
    "3.c.5_gestor da equipe de engenharia de dados": "gestor_eng_dados",
    "3.c.6_gestor da equipe de estudos, relatórios": "gestor_analise_dados",
    "3.c.7_gestor da equipe de Inteligência Artificial e Machine Learning": "gestor_ia",
    "3.c.8_Apesar de ser gestor ainda atuo na parte técnica": "gestor_ciencia_dados",
    "3.c.9_Gestão de projetos de dados": "gestor_projetos",
    "3.c.10_Gestão de produtos de dados": "gestor_produtos",
    "3.c.11_Gestão de pessoas": "gestor_pessoas",

    "3.d_desafios_como_gestor": "tres_maiores_desafios",
    "3.d.1_Contratar talentos": "contratar_novos_talentos",
    "3.d.2_Reter talentos": "reter_talentos",
    "3.d.3_Convencer a empresa a aumentar investimentos": "convencer_aumentar_invest_dados",
    "3.d.4_Gestão de equipes no ambiente remoto": "gestao_equipe_em_remoto",
    "3.d.5_Gestão de projetos envolvendo áreas multidisciplinares": "gestao_proj_multidisciplinaridades_empresa",
    "3.d.6_Organizar as informações com qualidade e confiabilidade": "organizar_info_resguard_quali_confiab",
    "3.d.7_Processar e armazenar um alto volume de dados": "processar_armazenar_big_data",
    "3.d.8_Gerar valor para as áreas de negócios": "gerar_valor_area_neg_atrav_estud_experim",
    "3.d.9_Desenvolver e manter modelos Machine Learning em produção": "desenv_e_mant_modelos_ml_prod",
    "3.d.10_Gerenciar a expectativa das áreas": "gerenciar_expectativ_areas_neg_rel_equipe_dados",
    "3.d.11_Garantir a manutenção dos projetos e modelos em produção": "manutencao_projetos_model_prod_meio_cresc_empresa",
    "3.d.12_Conseguir levar inovação para a empresa": "inovacao_atraves_area_dados",
    "3.d.13_Garantir (ROI) em projetos de dados": "retorno_roi_projeto_dados",
    "3.d.14_Dividir o tempo entre entregas técnicas e gestão": "dividir_tempo_entregas_tec_gestao",

    "3.e_ai_generativa_e_llm_é_uma_prioridade?": "ia_gen_eh_prioridade_empresa",
    "3.f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "tipo_uso_ia_generativa_llm_empresa",
    "3.f.1 Colaboradores usando AI generativa de forma independente e descentralizada": "colaboradores_uso_ia_descentra_independ",
    "3.f.2 Direcionamento centralizado do uso de AI generativa": "direcionamento_centralizado_ia_generativa",
    "3.f.3 Desenvolvedores utilizando Copilots": "dev_usando_copilot",
    "3.f.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais": "ia_gen_llm_melhorar_prod_ext",
    "3.f.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores": "ia_gen_llm_melhorar_prod_int_colaboradores",
    "3.f.6 IA Generativa e LLMs como principal frente do negócio": "ia_gen_llm_principal_frente_neg",
    "3.f.7 IA Generativa e LLMs não é prioridade": "ia_llm_nao_prioridade",
    "3.f.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa": "sem_opiniao_llm_ia_generativa",

    "3.g_motivos_para_não_usar_ai_generativa_e_llm": "motiv_empresa_nao_usar_ia_gen_llm",
    "3.g.1 Falta de compreensão dos casos de uso": "falta_compreensao_caso_uso",
    "3.g.2 Falta de confiabilidade das saídas (alucinação dos modelos)": "falta_confiab_saida_alucinacao_modelo",
    "3.g.3 Incerteza em relação a regulamentação": "incerteza_rel_regulamentacao",
    "3.g.4 Preocupações com segurança e privacidade de dados": "seguranca_privacidade_dados",
    "3.g.5 Retorno sobre investimento (ROI) não comprovado de IA Generativa": "retorno_roi_nao_comprovado_ia_gen",
    "3.g.6 Dados da empresa não estão prontos para uso de IA Generativa": "dados_empresa_nao_prep_ia_gen",
    "3.g.7 Falta de expertise ou falta de recursos": "falta_expertise_ou_recurso",
    "3.g.8 Alta direção da empresa não vê valor ou não vê como prioridade": "alta_direcao_nao_ve_valor_prioridade",
    "3.g.9 Preocupações com propriedade intelectual": "preocupacao_prop_intelectual",

    "4.a_funcao_de_atuacao": "nao_sendo_cargo_exerce_funcao",
    "4.a.1_atuacao_em_dados": "atuacao",
    "4.b_fontes_de_dados_(dia_a_dia)": "fontes_dados_analisou_trab",
    "4.b.1_Dados relacionais (estruturados em bancos SQL)": "banco_relacional",
    "4.b.2_Dados armazenados em bancos NoSQL": "banco_no_sql",
    "4.b.3_Imagens": "img",
    "4.b.4_Textos/Documentos": "text_doc",
    "4.b.5_Vídeos": "video",
    "4.b.6_Áudios": "audio",
    "4.b.7_Planilhas": "planilha",
    "4.b.8_Dados georeferenciados": "georefenciados",

    "4.c_fonte_de_dado_mais_usada": "fonte_dados_mais_utilizadas",
    "4.c.1_Dados relacionais (estruturados em bancos SQL)": "banco_relacional_fonte",
    "4.c.2_Dados armazenados em bancos NoSQL": "banco_no_sql_fonte",
    "4.c.3_Imagens": "img_fonte",
    "4.c.4_Textos/Documentos": "text_doc_fonte",
    "4.c.5_Vídeos": "video_fonte",
    "4.c.6_Áudios": "audio_fonte",
    "4.c.7_Planilhas": "planilha_fonte",
    "4.c.8_Dados georeferenciados": "georeferenciados_fonte",

    "4.d_linguagem_de_programacao_(dia_a_dia)": "linguagens_utilizadas_trab",
    "4.d.1_SQL": "sql",
    "4.d.2_R": "r",
    "4.d.3_Python": "python",
    "4.d.4_C/C++/C#": "ccc",
    "4.d.5_.NET": "net",
    "4.d.6_Java": "java",
    "4.d.7_Julia": "julia",
    "4.d.8_SAS/Stata": "sas_stat",
    "4.d.9_Visual Basic/VBA": "vba",
    "4.d.10_Scala": "scala",
    "4.d.11_Matlab": "matlab",
    "4.d.12_Rust": "rust",
    "4.d.13_PHP": "php",
    "4.d.14_JavaScript": "js",
    "4.d.15_Não utilizo nenhuma das linguagens listadas": "nao_utilizo_nenhuma",
    "4.e_linguagem_mais_usada": "linguagem_mais_utilizada_trab",
    "4.f_linguagem_preferida": "linguagem_preferida",

    "4.g_banco_de_dados_(dia_a_dia)": "banco_relacional_utilizado",
    "4.g.1_MySQL": "my_sql",
    "4.g.2_Oracle": "oracle",
    "4.g.3_SQL SERVER": "sql_server",
    "4.g.4_Amazon Aurora ou RDS": "amazon_aurora_rds",
    "4.g.5_DynamoDB": "dynamo_db",
    "4.g.6_CoachDB": "coach_db",
    "4.g.7_Cassandra": "cassandra",
    "4.g.8_MongoDB": "mongo_db",
    "4.g.9_MariaDB": "maria_db",
    "4.g.10_Datomic": "datomic",
    "4.g.11_S3": "s3",
    "4.g.12_PostgreSQL": "postgre_sql",
    "4.g.13_ElasticSearch": "elastic_search",
    "4.g.14_DB2": "db2",
    "4.g.15_Microsoft Access": "microsoft_acess",
    "4.g.16_SQLite": "sqlite",
    "4.g.17_Sybase": "sybase",
    "4.g.18_Firebase": "firebase",
    "4.g.19_Vertica": "vertica",
    "4.g.20_Redis": "redis",
    "4.g.21_Neo4J": "neo4j",
    "4.g.22_Google BigQuery": "google_big_query",
    "4.g.23_Google Firestore": "google_firestore",
    "4.g.24_Amazon Redshift": "amazon_redshift",
    "4.g.25_Amazon Athena": "amazon_athena",
    "4.g.26_Snowflake": "snowflake",
    "4.g.27_Databricks": "databricks",
    "4.g.28_HBase": "hbase",
    "4.g.29_Presto": "presto",
    "4.g.30_Splunk": "splunk",
    "4.g.31_SAP HANA": "sap_hana",
    "4.g.32_Hive": "hive",
    "4.g.33_Firebird": "firebird",

    "4.h_cloud_(dia_a_dia)": "cloud_dia_dia",
    "4.h.1_Amazon Web Services (AWS)": "aws",
    "4.h.2_Google Cloud (GCP)": "gcp",
    "4.h.3_Azure (Microsoft)": "azure",
    "4.h.4_Oracle Cloud": "oracle",
    "4.h.5_IBM": "ibm",
    "4.h.6_Servidores On Premise/Não utilizamos Cloud": "on_premise",
    "4.h.7_Cloud Própria": "cloud_proprietaria",
    "4.i_cloud_preferida": "cloud_preferida",

    "4.j_ferramenta_de_bi_(dia_a_dia)": "ferramenta_bi_diaria",
    "4.j.1_Microsoft PowerBI": "pbi",
    "4.j.2_Qlik View/Qlik Sense": "qlik_view_sense",
    "4.j.3_Tableau": "tableau",
    "4.j.4_Metabase": "metabase",
    "4.j.5_Superset": "superset",
    "4.j.6_Redash": "redash",
    "4.j.7_Looker": "looker",
    "4.j.8_Looker Studio(Google Data Studio)": "looker_studio",
    "4.j.9_Amazon Quicksight": "amazon_quicksight",
    "4.j.10_Alteryx": "alteryx",
    "4.j.11_SAP Business Objects/SAP Analytics": "sap_business_objects",
    "4.j.12_Oracle Business Intelligence": "oracle_bi",
    "4.j.13_Salesforce/Einstein Analytics": "salesforce_einstein_analytics",
    "4.j.14_SAS Visual Analytics": "sas_visual_analytics",
    "4.j.15_Grafana": "grafana",
    "4.j.16_Pentaho": "pentaho",
    "4.j.17_Fazemos todas as análises utilizando apenas Excel ou planilhas do google": "excel_planilha_apenas",
    "4.j.18_Não utilizo nenhuma ferramenta de BI no trabalho": "nenhuma_ferramenta_bi",
    "4.k_ferramenta_de_bi_preferida": "ferramenta_bi_favorita",

    "4.l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "tipo_uso_ia_llm_empresa",
    "4.l.1 Colaboradores usando AI generativa de forma independente e descentralizada": "ia_llm_empresa_descentralizada_independente",
    "4.l.2 Direcionamento centralizado do uso de AI generativa": "ia_llm_empresa_uso_centralizado",
    "4.l.3 Desenvolvedores utilizando Copilots": "ia_llm_empresa_uso_copilot",
    "4.l.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais": "ia_llm_empresa_prod_externo",
    "4.l.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores": "ia_llm_empresa_prod_interno",
    "4.l.6 IA Generativa e LLMs como principal frente do negócio": "ia_llm_empresa_frente_negoc",
    "4.l.7 IA Generativa e LLMs não é prioridade": "ia_llm_empresa_nao_prioridade",
    "4.l.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa": "ia_llm_empresa_sem_opiniao",

    "4.m_usa_chatgpt_ou_copilot_no_trabalho?": "utilizacao_chat_gpt_llm",
    "4.m.1 Não uso soluções de AI Generativa com foco em produtividade": "nao_uso_ia_gen_produtividade",
    "4.m.2 Uso soluções gratuitas de AI Generativa com foco em produtividade": "uso_ia_gen_gratuita_produtividade",
    "4.m.3 Uso e pago pelas soluções de AI Generativa com foco em produtividade": "uso_pago_ia_gen_produtividade",
    "4.m.4 A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade": "uso_pago_ia_gen_empresa_paga",
    "4.m.5 Uso soluções do tipo Copilot": "uso_copilot",

    "5.a_objetivo_na_area_de_dados": "obj_area_dados",
    "5.b_oportunidade_buscada": "oportunidade_almejada",
    "5.c_tempo_em_busca_de_oportunidade": "tempo_busca_oport_area_dados",
    "5.d_experiencia_em_processos_seletivos": "busca_empreg_area_dados",

    "6.a_rotina_como_de": "rotina_engenheiro_dados",
    "6.a.1_Desenvolvo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.": "pipeline_dados_ling_prog",
    "6.a.2_Realizo construções de ETL's em ferramentas como Pentaho, Talend, Dataflow etc.": "etl_pentaho_talent_etc",
    "6.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "relatorios_sql_exportados",
    "6.a.4_Atuo na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.": "integracao_dados_plataformas_dados",
    "6.a.5_Modelo soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.": "criacao_componentes_ingestao_dados",
    "6.a.6_Desenvolvo/cuido da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.": "manutencao_criacao_rep_dados_streaming",
    "6.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "modelagem_dados_dw_data_mart",
    "6.a.8_Cuido da qualidade dos dados, metadados e dicionário de dados.": "quali_dados_dicionario",
    "6.a.9_Nenhuma das opções listadas refletem meu dia a dia.": "nenhuma_opcao_engenheiro_dados",

    "6.b_ferramentas_etl_de": "ferramenta_etl_utilizado_engenheiro_dados",
    "6.b.1_Scripts Python": "script_python",
    "6.b.2_SQL & Stored Procedures": "sql_stored_procedures",
    "6.b.3_Apache Airflow": "apache_airflow",
    "6.b.4_Apache NiFi": "apache_nifi",
    "6.b.5_Luigi": "luigi",
    "6.b.6_AWS Glue": "aws_glue",
    "6.b.7_Talend": "talend",
    "6.b.8_Pentaho": "engenheiro_dados_pentaho",
    "6.b.9_Alteryx": "engenheiro_dados_alteryx",
    "6.b.10_Stitch": "stitch",
    "6.b.11_Fivetran": "fivetran",
    "6.b.12_Google Dataflow": "google_dataflow",
    "6.b.13_Oracle Data Integrator": "oracle_data_integrator",
    "6.b.14_IBM DataStage": "ibm_data_stage",
    "6.b.15_SAP BW ETL": "sap_bw_etl",
    "6.b.16_SQL Server Integration Services (SSIS)": "SSIS",
    "6.b.17_SAS Data Integration": "sas_data_integration",
    "6.b.18_Qlik Sense": "qlik_sense",
    "6.b.19_Knime": "knime",
    "6.b.20_Databricks": "engenheiro_dados_databricks",
    "6.b.21_Não utilizo ferramentas de ETL": "engenheiro_dados_sem_ferramenta_etl",

    "6.c_possui_data_lake": "empresa_possui_datalake",
    "6.d_tecnologia_data_lake": "tecnologia_utilizada_datalake",
    "6.e_possui_data_warehouse": "empresa_possui_dw",
    "6.f_tecnologia_data_warehouse": "tecnologia_utilizada_dw",
    "6.g_ferramentas_de_qualidade_de_dados_(dia_a_dia)": "ferramentas_meta_dados_qualidade_dados_trabalho",
    "6.h_maior_tempo_gasto_como_de": "maior_parte_tempo_gasta",
    "6.h.1_Desenvolvendo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.": "pipelines_com_codigo",
    "6.h.2_Realizando construções de ETL\\s em ferramentas como Pentaho, Talend, Dataflow etc.": "etl_com_ferramentas_bi",
    "6.h.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "exportacao_rel_para_area_neg",
    "6.h.4_Atuando na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.": "integracao_fonte_dados_diferentes_tmp_gasto",
    "6.h.5_Modelando soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.": "modelagem_arquitetura_dados_tmp_gasto",
    "6.h.6_Desenvolvendo/cuidando da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.": "dados_streaming_data_lake_lakehouse",
    "6.h.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "modelagem_dados_dw_data_mart",
    "6.h.8_Cuidando da qualidade dos dados, metadados e dicionário de dados.": "qualidade_dados_tmp_gasto",
    "6.h.9_Nenhuma das opções listadas refletem meu dia a dia.": "nenhuma_das_opcoes_tmp_gasto",

    "7.a_rotina_como_da": "analise_dados_rotina",
    "7.a.1_Processo e analiso dados utilizando linguagens de programação como Python, R etc.": "analise_processamento_linguagem_prog",
    "7.a.2_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.": "dashboard_ferramentas_bi",
    "7.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "exportacao_rel_area_neg_analista_dados",
    "7.a.4_Utilizo API\\s para extrair dados e complementar minhas análises.": "extracao_dados_api",
    "7.a.5_Realizo experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.": "estudo_experimentos_util_estatistica",
    "7.a.6_Desenvolvo/cuido da manutenção de ETL\\s utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": "manutencao_etl_utilizando_ferramenta_etl_analista",
    "7.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts etc.": "modelagem_dw_data_mart_analista",
    "7.a.8_Desenvolvo/cuido da manutenção de planilhas para atender as áreas de negócio.": "manutencao_planilha_area_negocio",
    "7.a.9_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.": "utilizacao_ferramenta_avancada_estatistica",
    "7.a.10_Nenhuma das opções listadas refletem meu dia a dia.": "nenhuma_das_opcoes_analista",

    "7.b_ferramentas_etl_da": "ferramental_analista_dados",
    "7.b.1_Scripts Python": "python_analista",
    "7.b.2_SQL & Stored Procedures": "stored_procedures_analista",
    "7.b.3_Apache Airflow": "apache_airflow_analista",
    "7.b.4_Apache NiFi": "apache_nifi_analista",
    "7.b.5_Luigi": "luigi_analista",
    "7.b.6_AWS Glue": "aws_glue_analista",
    "7.b.7_Talend": "talent_analista",
    "7.b.8_Pentaho": "pentaho_analista",
    "7.b.9_Alteryx": "alteryx_analista",
    "7.b.10_Stitch": "stitch_analista",
    "7.b.11_Fivetran": "fivetran_analista",
    "7.b.12_Google Dataflow": "google_dataflow_analista",
    "7.b.13_Oracle Data Integrator": "oracle_data_integrator_analista",
    "7.b.14_IBM DataStage": "ibm_data_stage_analista",
    "7.b.15_SAP BW ETL": "sap_bw_etl_analista",
    "7.b.16_SQL Server Integration Services (SSIS)": "ssis_analista",
    "7.b.17_SAS Data Integration": "sas_data_integration_analista",
    "7.b.18_Qlik Sense": "qlik_sense_analista",
    "7.b.19_Knime": "knime_analista",
    "7.b.20_Databricks": "databricks_analista",
    "7.b.21_Não utilizo ferramentas de ETL": "nao_utilizo_ferramentas_etl_analista",

    "7.c_ferramentas_autonomia_area_de_negocios": "ferramentas_empresa_usadas_mais_autonomia_area_neg",
    "7.c.1_Ferramentas de AutoML como H2O.ai, Data Robot, BigML etc.": "ferramentas_auto_ml",
    "7.c.2_\"Point and Click\" Analytics como Alteryx, Knime, Rapidminer etc.": "point_click",
    "7.c.3_Product metricts & Insights como Mixpanel, Amplitude, Adobe Analytics.": "product_metrics_insight",
    "7.c.4_Ferramentas de análise dentro de ferramentas de CRM como Salesforce Einstein Anaytics ou Zendesk dashboards.": "ferramentas_analise_dados_dentro_crm",
    "7.c.5_Minha empresa não utiliza essas ferramentas.": "empresa_nao_utiliza_ferramenta_autonomia",
    "7.c.6_Não sei informar.": "nao_sei_informar_ferramenta_autonomia_area_neg",

    "7.d_maior_tempo_gasto_como_da": "maior_parte_tempo_gasto_analista",
    "7.d.1_Processando e analisando dados utilizando linguagens de programação como Python, R etc.": "dashboards_script_analista",
    "7.d.2_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.": "dashboards_ferramenta_bi_analista",
    "7.d.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "extracao_rel_sql_analista",
    "7.d.4_Utilizando API's para extrair dados e complementar minhas análises.": "extracao_api_analista",
    "7.d.5_Realizando experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.": "estudo_uso_estatistica_analista",
    "7.d.6_Desenvolvendo/cuidando da manutenção de ETL's utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": "criacao_manutencao_etl_analista",
    "7.d.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "modelagem_dados_dw_data_mart_analista",
    "7.d.8_Desenvolvendo/cuidando da manutenção de planilhas para atender as áreas de negócio.": "manutencao_planilhas_analista",
    "7.d.9_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.": "ferramentas_avanc_estatistica_analista",
    "7.d.10_Nenhuma das opções listadas refletem meu dia a dia.": "nenhuma_opcao_listada_atv_analista",

    "8.a_rotina_como_ds": "rotina_trabalho_cientista",
    "8.a.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.": "uso_adhoc_modelo_preditivo",
    "8.a.2_Sou responsável pela coleta e limpeza dos dados que uso para análise e modelagem.": "coleta_limpeza_modelagem_analise",
    "8.a.3_Sou responsável por entrar em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.": "analise_requisitos_area_negocio",
    "8.a.4_Desenvolvo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).": "dev_modelo_ml",
    "8.a.5_Sou responsável por colocar modelos em produção, criar os pipelines de dados, APIs de consumo e monitoramento.": "deploy_modelo_prod_criacao_api_monitoramento",
    "8.a.6_Cuido da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.": "manutencao_modelo_ml_producao_melhorias",
    "8.a.7_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc": "construcao_dashboard_ferramenta_bi_analista",
    "8.a.8_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.": "ferramenta_avanc_estatistica_analise_ajust_modelo",
    "8.a.9_Crio e dou manutenção em ETLs, DAGs e automações de pipelines de dados.": "criacao_manun_etl_dags_automacoes",
    "8.a.10_Crio e gerencio soluções de Feature Store e cultura de MLOps.": "criacao_gerencia_feature_store_mlops",
    "8.a.11_Sou responsável por criar e manter a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)": "infra_modelos_rodam_em_clusters_api_etc",
    "8.a.12_Treino e aplico LLM's para solucionar problemas de negócio.": "criacao_treinamento_llm",

    "8.b_tecnicas_e_metodos_ds": "tecnica_metodo_utilizado_cientista_trabalho",
    "8.b.1_Utilizo modelos de regressão (linear, logística, GLM).": "regressao_logistica_glm",
    "8.b.2_Utilizo redes neurais ou modelos baseados em árvore para criar modelos de classificação.": "redes_neurais_odelo_arvore",
    "8.b.3_Desenvolvo sistemas de recomendação (RecSys).": "sistema_recomendacao",
    "8.b.4_Utilizo métodos estatísticos Bayesianos para analisar dados.": "metodo_estatistico_bayesiano_analise",
    "8.b.5_Utilizo técnicas de NLP (Natural Language Processing) para análisar dados não-estruturados.": "NLP_dados_nao_estruturados",
    "8.b.6_Utilizo métodos estatísticos clássicos (Testes de hipótese, análise multivariada, sobrevivência, dados longitudinais, inferência estatistica) para analisar dados.": "metodos_estatisticos_classicos",
    "8.b.7_Utilizo cadeias de Markov ou HMM\\s para realizar análises de dados.": "cadeias_markov_hmm_analise_dados",
    "8.b.8_Desenvolvo técnicas de Clusterização (K-means, Spectral, DBScan etc).": "tecnicas_clusterizacao",
    "8.b.9_Realizo previsões através de modelos de Séries Temporais (Time Series).": "modelos_preditivos",
    "8.b.10_Utilizo modelos de Reinforcement Learning (aprendizado por reforço).": "aprendizado_reforco",
    "8.b.11_Utilizo modelos de Machine Learning para detecção de fraude.": "ml_deteccao_fraude",
    "8.b.12_Utilizo métodos de Visão Computacional.": "visao_computacional",
    "8.b.13_Utilizo modelos de Detecção de Churn.": "modelo_deteccao_churn",
    "8.b.14_Utilizo LLM's para solucionar problemas de negócio.": "llm_cientista_dados_uso",

    "8.c_tecnologias_ds": "tecnologias_diarias_cientista",
    "8.c.1_Ferramentas de BI (PowerBI, Looker, Tableau, Qlik etc).": "ferramenta_bi_cientista",
    "8.c.2_Planilhas (Excel, Google Sheets etc).": "planilha_excel_cientista",
    "8.c.3_Ambientes de desenvolvimento local (R-studio, JupyterLab, Anaconda).": "ambiente_local_jupyter_anaconda",
    "8.c.4_Ambientes de desenvolvimento na nuvem (Google Colab, AWS Sagemaker, Kaggle Notebooks etc).": "dev_nuvem",
    "8.c.5_Ferramentas de AutoML (Datarobot, H2O, Auto-Keras etc).": "auto_ml_cientista",
    "8.c.6_Ferramentas de ETL (Apache Airflow, NiFi, Stitch, Fivetran, Pentaho etc).": "ferramenta_etl_cientista",
    "8.c.7_Plataformas de Machine Learning (TensorFlow, Azure Machine Learning, Kubeflow etc).": "plataforma_ml_cientista",
    "8.c.8_Feature Store (Feast, Hopsworks, AWS Feature Store, Databricks Feature Store etc).": "feature_store_cientista",
    "8.c.9_Sistemas de controle de versão (Github, DVC, Neptune, Gitlab etc).": "controle_de_versao_cientista",
    "8.c.10_Plataformas de Data Apps (Streamlit, Shiny, Plotly Dash etc).": "data_apps_cientista",
    "8.c.11_Ferramentas de estatística avançada como SPSS, SAS, etc.": "ferramenta_estatistica_avanc_cientista",

    "8.d_maior_tempo_gasto_como_ds": "tempo_gasto_cientista",
    "8.d.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.": "ad_hoc_hipoteses_tmp_gasto_cientista",
    "8.d.2_Coletando e limpando dos dados que uso para análise e modelagem.": "coleta_limpeza_cientista",
    "8.d.3_Entrando em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.": "analise_requisitos_tmp_gasto_cientista",
    "8.d.4_Desenvolvendo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).": "dev_ml_prod_tmp_gasto_cientista",
    "8.d.5_Colocando modelos em produção, criando os pipelines de dados, APIs de consumo e monitoramento.": "manutencao_ml_prod_tmp_gasto_cientista",
    "8.d.6_Cuidando da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.": "manutencao_ml_em_prod_monitoramento_ajuste_tmp_gasto_cientista",
    "8.d.7_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc.": "dashboards_ferramenta_bi_tmp_gasto_cientista",
    "8.d.8_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.": "ferramenta_avancadas_estatistica_analista_tmp_gasto_cientista",
    "8.d.9_Criando e dando manutenção em ETLs, DAGs e automações de pipelines de dados.": "manutencao_etl_dag_tmp_gasto_cientista",
    "8.d.10_Criando e gerenciando soluções de Feature Store e cultura de MLOps.": "feature_store_ml_ops_tmp_gasto_cientista",
    "8.d.11_Criando e mantendo a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)": "infra_modelo_clusters_servidores_tmp_gasto_cientista",
    "8.d.12_Treinando e aplicando LLM's para solucionar problemas de negócio.": "LLM_tmp_gasto_cientista"
}

In [10]:
df_state_data_2024 = ler_df_csv(sessao, caminho_2024)
df_state_data_2024 = renomear_colunas(df_state_data_2024, novos_nomes_colunas_2024)
df_state_data_2024 = processar_dataframe(df_state_data_2024)

In [11]:
exportar_df_para_csv(df_state_data_2024, "bronze_dw_state_data_2024.csv")

# Criação da camada bronze do DW da pesquisa para 2025

In [16]:
novos_nomes_colunas_2025 = {
    '0.a_token': 'token_user',
    '0.d_data/hora_envio': 'data_hora_envio',
    '1.a_idade': 'idade',
    '1.a.1_faixa_idade': 'faixa_etaria',
    '1.b_genero': 'genero',
    '1.c_cor/raca/etnia': 'cor_raca_etnia',
    '1.d_pcd': 'pcd',
    '1.e_experiencia_profissional_prejudicada': 'exp_prof_prejud',
    '1.e.1_Não acredito que minha experiência profissional seja afetada': 'exp_prof_nao_prejud',
    '1.e.2_Sim, devido a minha Cor/Raça/Etnia': 'exp_prof_prejud_cor_raca_etnia',
    '1.e.3_Sim, devido a minha identidade de gênero': 'exp_prof_prejud_ident_gen',
    '1.e.4_Sim, devido ao fato de ser PCD': 'exp_prof_prejud_pcd',
    '1.f_aspectos_prejudicados': 'aspectos_prejud',
    '1.f.1_Quantidade de oportunidades de emprego/vagas recebidas': 'qtd_oportunidades_vagas_emprego_receb',
    '1.f.2_Senioridade das vagas recebidas em relação à sua experiência': 'senioridade_vagas_recebidas_rel_experiencia',
    '1.f.3_Aprovação em processos seletivos/entrevistas': 'aprov_processos_seletivos_entrevistas',
    '1.f.4_Oportunidades de progressão de carreira': 'oportunidades_progresso_carreira',
    '1.f.5_Velocidade de progressão de carreira': 'vel_progressao_carreira',
    '1.f.6_Nível de cobrança no trabalho/Stress no trabalho': 'nvl_cobranca_e_stress_trab',
    '1.f.7_Atenção dada pelas pessoas diante das minhas opiniões e ideias': 'atencao_opiniao_ideias',
    '1.f.8_Relação com outras pessoas da empresa, em momentos de trabalho': 'rel_outros_membros_empresa_em_trabalho',
    '1.f.9_Relação com outras pessoas da empresa, em momentos de integração e outros momentos fora do trabalho': 'rel_outros_membros_empresa_integracao_fora_trab',
    '1.g_vive_no_brasil': 'vive_brasil',
    '1.h_pais_onde_mora': 'pais',
    '1.i_estado_onde_mora': 'estado',
    '1.i.1_uf_onde_mora': 'uf_moradia',
    '1.i.2_regiao_onde_mora': 'regiao',
    '1.j_vive_no_estado_de_formacao': 'vive_estado_formacao',
    '1.k_estado_de_origem': 'estado_origem',
    '1.k.1_uf_de_origem': 'uf',
    '1.k.2_regiao_de_origem': 'regiao_origem',
    '1.l_nivel_de_ensino': 'nvl_ensino',
    '1.m_área_de_formação': 'area_formacao',
    '2.a_situação_de_trabalho': 'sit_atual_trab',
    '2.b_setor': 'setor',
    '2.c_numero_de_funcionarios': 'n_funcionarios',
    '2.d_atua_como_gestor': 'gestor',
    '2.e_cargo_como_gestor': 'cargo_gestor',
    '2.f_cargo_atual': 'cargo_atual',
    '2.g_nivel': 'nvl',
    '2.h_faixa_salarial': 'faixa_salarial',
    '2.i_tempo_de_experiencia_em_dados': 'tempo_exp_dados',
    '2.j_tempo_de_experiencia_em_ti': 'tempo_exp_ti',
    '2.k_satisfeito_atualmente': 'satisfacao_empresa_atual',
    '2.l.1_Remuneração/Salário': 'remuneracao_salario',
    '2.l.2_Benefícios': 'beneficios',
    '2.l.3_Propósito do trabalho e da empresa': 'proposito_trabalho_empresa',
    '2.l.4_Flexibilidade de trabalho remoto': 'flexibilidade_trab_remoto',
    '2.l.5_Ambiente e clima de trabalho': 'ambiente_clima_trabalho',
    '2.l.6_Oportunidade de aprendizado e trabalhar com referências': 'oport_aprendizado_trab_ref_area',
    '2.l.7_Oportunidades de crescimento': 'plano_carreira_oport_cresc_prof',
    '2.l.8_Maturidade da empresa em termos de tecnologia e dados': 'maturidade_empresa_dados_tec',
    '2.l.9_Relação com os gestores e líderes': 'qualidade_lideres_gestores',
    '2.l.10_Reputação que a empresa tem no mercado': 'rep_empresa_mercado',
    '2.l.11_Gostaria de trabalhar em outra área': 'trab_outra_area_atuacao',
    '2.l_motivo_insatisfacao': 'motivo_insatis_empresa_atual',
    '2.m_participou_de_entrevistas_ultimos_6m': 'participou_entrevistas_ult_6_meses',
    '2.n_planos_de_mudar_de_emprego_6m': 'mudar_emprego_prox_6_meses',
    '2.o_criterios_para_escolha_de_emprego': 'principais_criterios_levados_decidir_trab',
    '2.o.1_Remuneração/Salário': 'remuneracao_salario',
    '2.o.2_Benefícios': 'beneficios',
    '2.o.3_Propósito do trabalho e da empresa': 'proposito_trabalho_empresa',
    '2.o.4_Flexibilidade de trabalho remoto': 'flexibilidade_trab_remoto',
    '2.o.5_Ambiente e clima de trabalho': 'ambiente_clima_trabalho',
    '2.o.6_Oportunidade de aprendizado e trabalhar com referências': 'oport_aprendizado_trab_ref_area',
    '2.o.7_Plano de carreira e oportunidades de crescimento': 'plano_carreira_oport_cresc_prof',
    '2.o.8_Maturidade da empresa em termos de tecnologia e dados': 'maturidade_empresa_dados_tec',
    '2.o.9_Qualidade dos gestores e líderes': 'qualidade_lideres_gestores',
    '2.o.10_Reputação que a empresa tem no mercado': 'rep_empresa_mercado',
    '2.p_empresa_passou_por_layoff_em_2025': 'layoff_2025',
    '2.q_modelo_de_trabalho_atual': 'forma_trabalho',
    '2.r_modelo_de_trabalho_ideal': 'forma_trabalho_ideal',
    '2.s_atitude_em_caso_de_retorno_presencial': 'atitude_perante_100_presencial',
    '3.a_numero_de_pessoas_em_dados': 'num_prof_dados_atuando_empresa_hoje',
    '3.b_cargos_no_time_de_dados_da_empresa': 'papeis_cargos_no_time_empresa',
    '3.b.1_Analytics Engineer': 'engenheiro_analytics',
    '3.b.2_Engenharia de Dados/Data Engineer': 'engenheiro_dados',
    '3.b.3_Analista de Dados/Data Analyst': 'analista_dados',
    '3.b.4_Cientista de Dados/Data Scientist': 'cientista_dados',
    '3.b.5_Database Administrator/DBA': 'dba',
    '3.b.6_Analista de Business Intelligence/BI': 'analista_bi',
    '3.b.7_Arquiteto de Dados/Data Architect': 'arquiteto_dados',
    '3.b.8_Data Product Manager/DPM': 'data_product_manager',
    '3.b.9_Business Analyst': 'analista_business',
    '3.b.10_ML Engineer/AI Engineer': 'engenheiro_ml_ia',
    '3.c_responsabilidades_como_gestor': 'resp_rotina_atual_como_gestor',
    '3.c.1_Pensar na visão de longo prazo de dados': 'visao_longo_prazo_dados_empresa',
    '3.c.2_Organização de treinamentos e iniciativas': 'organizacao_treinamentos_iniciat_para_maior_maturidade_analitica',
    '3.c.3_Atração, seleção e contratação': 'contratacao_area_dados',
    '3.c.4_Decisão sobre contratação de ferramentas': 'contratacao_ferramentas_dados',
    '3.c.5_gestor da equipe de engenharia de dados': 'gestor_eng_dados',
    '3.c.6_gestor da equipe de estudos, relatórios': 'gestor_analise_dados',
    '3.c.7_gestor da equipe de Inteligência Artificial e Machine Learning': 'gestor_ia',
    '3.c.8_Apesar de ser gestor ainda atuo na parte técnica': 'gestor_ciencia_dados',
    '3.c.9_Gestão de projetos de dados': 'gestor_projetos',
    '3.c.10_Gestão de produtos de dados': 'gestor_produtos',
    '3.c.11_Gestão de pessoas': 'gestor_pessoas',
    '3.d_desafios_como_gestor': 'tres_maiores_desafios',
    '3.d.1_Contratar talentos': 'contratar_novos_talentos',
    '3.d.2_Reter talentos': 'reter_talentos',
    '3.d.3_Convencer a empresa a aumentar investimentos': 'convencer_aumentar_invest_dados',
    '3.d.4_Gestão de equipes no ambiente remoto': 'gestao_equipe_em_remoto',
    '3.d.5_Gestão de projetos envolvendo áreas multidisciplinares': 'gestao_proj_multidisciplinaridades_empresa',
    '3.d.6_Organizar as informações com qualidade e confiabilidade': 'organizar_info_resguard_quali_confiab',
    '3.d.7_Processar e armazenar um alto volume de dados': 'processar_armazenar_big_data',
    '3.d.8_Gerar valor para as áreas de negócios': 'gerar_valor_area_neg_atrav_estud_experim',
    '3.d.9_Desenvolver e manter modelos Machine Learning em produção': 'desenv_e_mant_modelos_ml_prod',
    '3.d.10_Gerenciar a expectativa das áreas': 'gerenciar_expectativ_areas_neg_rel_equipe_dados',
    '3.d.11_Garantir a manutenção dos projetos e modelos em produção': 'manutencao_projetos_model_prod_meio_cresc_empresa',
    '3.d.12_Conseguir levar inovação para a empresa': 'inovacao_atraves_area_dados',
    '3.d.13_Garantir (ROI) em projetos de dados': 'retorno_roi_projeto_dados',
    '3.d.14_Dividir o tempo entre entregas técnicas e gestão': 'dividir_tempo_entregas_tec_gestao',
    '3.e_ai_generativa_e_llm_é_uma_prioridade?': 'ia_gen_eh_prioridade_empresa',
    '3.f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa': 'tipo_uso_ia_generativa_llm_empresa',
    '3.f.1 Colaboradores usando AI generativa de forma independente e descentralizada': 'colaboradores_uso_ia_descentra_independ',
    '3.f.2 Direcionamento centralizado do uso de AI generativa': 'direcionamento_centralizado_ia_generativa',
    '3.f.3 Desenvolvedores utilizando Copilots': 'dev_usando_copilot',
    '3.f.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais': 'ia_gen_llm_melhorar_prod_ext',
    '3.f.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores': 'ia_gen_llm_melhorar_prod_int_colaboradores',
    '3.f.6 IA Generativa e LLMs como principal frente do negócio': 'ia_gen_llm_principal_frente_neg',
    '3.f.7 IA Generativa e LLMs não é prioridade': 'ia_llm_nao_prioridade',
    '3.f.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa': 'sem_opiniao_llm_ia_generativa',
    '3.g_empresa_está_conseguindo_ter_bons_resultados_com_llms': 'llms_bom_resultado',
    '3.h_motivos_para_não_usar_ai_generativa_e_llm': 'motiv_empresa_nao_usar_ia_gen_llm',
    '3.h.1 Falta de compreensão dos casos de uso': 'falta_compreensao_caso_uso',
    '3.h.2 Falta de confiabilidade das saídas (alucinação dos modelos)': 'falta_confiab_saida_alucinacao_modelo',
    '3.h.3 Incerteza em relação a regulamentação': 'incerteza_rel_regulamentacao',
    '3.h.4 Preocupações com segurança e privacidade de dados': 'seguranca_privacidade_dados',
    '3.h.5 Retorno sobre investimento (ROI) não comprovado de IA Generativa': 'retorno_roi_nao_comprovado_ia_gen',
    '3.h.6 Dados da empresa não estão prontos para uso de IA Generativa': 'dados_empresa_nao_prep_ia_gen',
    '3.h.7 Falta de expertise ou falta de recursos': 'falta_expertise_ou_recurso',
    '3.h.8 Alta direção da empresa não vê valor ou não vê como prioridade': 'alta_direcao_nao_ve_valor_prioridade',
    '3.h.9 Preocupações com propriedade intelectual': 'preocupacao_prop_intelectual',
    '4.a.1_atuacao_em_dados': 'atuacao',
    '4.a_funcao_de_atuacao': 'nao_sendo_cargo_exerce_funcao',
    '4.b_fontes_de_dados_(dia_a_dia)': 'fontes_dados_analisou_trab',
    '4.b.1_Dados relacionais (estruturados em bancos SQL)': 'banco_relacional',
    '4.b.2_Dados armazenados em bancos NoSQL': 'banco_no_sql',
    '4.b.3_Imagens': 'img',
    '4.b.4_Textos/Documentos': 'text_doc',
    '4.b.5_Vídeos': 'video',
    '4.b.6_Áudios': 'audio',
    '4.b.7_Planilhas': 'planilha',
    '4.b.8_Dados georeferenciados': 'georefenciados',
    '4.c_linguagem_preferida': 'linguagem_preferida',
    '4.c.1_SQL': 'sql',
    '4.c.2_R': 'r',
    '4.c.3_Python': 'python',
    '4.c.4_C/C++/C#': 'ccc',
    '4.c.5_Julia': 'julia',
    '4.c.6_Visual Basic/VBA': 'vba',
    '4.c.7_Scala': 'scala',
    '4.c.8_DAX': 'DAX',
    '4.c.9_Rust': 'rust',
    '4.c.10_Não utilizo nenhuma das linguagens listadas': 'nao_utilizo_nenhuma',
    '4.d_banco_de_dados_(dia_a_dia)': 'banco_relacional_utilizado',
    '4.d.1_MySQL': 'my_sql',
    '4.d.2_Oracle': 'oracle',
    '4.d.3_SQL SERVER': 'sql_server',
    '4.d.4_Amazon Aurora ou RDS': 'amazon_aurora_rds',
    '4.d.5_DynamoDB': 'dynamo_db',
    '4.d.6_CoachDB': 'coach_db',
    '4.d.7_Cassandra': 'cassandra',
    '4.d.8_MongoDB': 'mongo_db',
    '4.d.9_MariaDB': 'maria_db',
    '4.d.10_Datomic': 'datomic',
    '4.d.11_S3': 's3',
    '4.d.12_PostgreSQL': 'postgre_sql',
    '4.d.13_ElasticSearch': 'elastic_search',
    '4.d.14_DB2': 'db2',
    '4.d.15_Microsoft Access': 'microsoft_acess',
    '4.d.16_SQLite': 'sqlite',
    '4.d.17_Sybase': 'sybase',
    '4.d.18_Firebase': 'firebase',
    '4.d.19_Vertica': 'vertica',
    '4.d.20_Redis': 'redis',
    '4.d.21_Neo4J': 'neo4j',
    '4.d.22_Google BigQuery': 'google_big_query',
    '4.d.23_Google Firestore': 'google_firestore',
    '4.d.24_Amazon Redshift': 'amazon_redshift',
    '4.d.25_Amazon Athena': 'amazon_athena',
    '4.d.26_Snowflake': 'snowflake',
    '4.d.27_Databricks': 'databricks',
    '4.d.28_HBase': 'hbase',
    '4.d.29_Presto': 'presto',
    '4.d.30_Splunk': 'splunk',
    '4.d.31_SAP HANA': 'sap_hana',
    '4.d.32_Hive': 'hive',
    '4.d.33_Firebird': 'firebird',
    '4.e_cloud_(dia_a_dia)': 'cloud_dia_dia',
    '4.e.1_Amazon Web Services (AWS)': 'aws',
    '4.e.2_Google Cloud (GCP)': 'gcp',
    '4.e.3_Azure (Microsoft)': 'azure',
    '4.e.4_Oracle Cloud': 'oracle',
    '4.e.5_IBM': 'ibm',
    '4.e.6_Servidores On Premise/Não utilizamos Cloud': 'on_premise',
    '4.e.7_Cloud Própria': 'cloud_proprietaria',
    '4.f_cloud_preferida': 'cloud_preferida',
    '4.g_ferramenta_de_bi_(dia_a_dia)': 'ferramenta_bi_diaria',
    '4.g.1_Microsoft PowerBI': 'pbi',
    '4.g.2_Qlik View/Qlik Sense': 'qlik_view_sense',
    '4.g.3_Tableau': 'tableau',
    '4.g.4_Metabase': 'metabase',
    '4.g.5_Superset': 'superset',
    '4.g.6_Redash': 'redash',
    '4.g.7_Looker': 'looker',
    '4.g.8_Looker Studio(Google Data Studio)': 'looker_studio',
    '4.g.9_Amazon Quicksight': 'amazon_quicksight',
    '4.g.10_Alteryx': 'alteryx',
    '4.g.11_SAP Business Objects/SAP Analytics': 'sap_business_objects',
    '4.g.12_Oracle Business Intelligence': 'oracle_bi',
    '4.g.13_Salesforce/Einstein Analytics': 'salesforce_einstein_analytics',
    '4.g.14_SAS Visual Analytics': 'sas_visual_analytics',
    '4.g.15_Grafana': 'grafana',
    '4.g.16_Pentaho': 'pentaho',
    '4.g.17_Fazemos todas as análises utilizando apenas Excel ou planilhas do google': 'excel_planilha_apenas',
    '4.g.18_Não utilizo nenhuma ferramenta de BI no trabalho': 'nenhuma_ferramenta_bi',
    '4.h_ferramenta_de_bi_preferida': 'ferramenta_bi_favorita',
    '4.i_tipo_de_uso_de_ai_generativa_e_llm_na_empresa': 'tipo_uso_ia_llm_empresa',
    '4.i.1 Colaboradores usando AI generativa de forma independente e descentralizada': 'ia_llm_empresa_descentralizada_independente',
    '4.i.2 Direcionamento centralizado do uso de AI generativa': 'ia_llm_empresa_uso_centralizado',
    '4.i.3 Desenvolvedores utilizando Copilots': 'ia_llm_empresa_uso_copilot',
    '4.i.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais': 'ia_llm_empresa_prod_externo',
    '4.i.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores': 'ia_llm_empresa_prod_interno',
    '4.i.6 IA Generativa e LLMs como principal frente do negócio': 'ia_llm_empresa_frente_negoc',
    '4.i.7 IA Generativa e LLMs não é prioridade': 'ia_llm_empresa_nao_prioridade',
    '4.i.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa': 'ia_llm_empresa_sem_opiniao',
    '4.j_usa_chatgpt_ou_copilot_no_trabalho?': 'utilizacao_chat_gpt_llm',
    '4.j.1 Não uso soluções de AI Generativa com foco em produtividade': 'nao_uso_ia_gen_produtividade',
    '4.j.2 Uso soluções gratuitas de AI Generativa com foco em produtividade': 'uso_ia_gen_gratuita_produtividade',
    '4.j.3 Uso e pago pelas soluções de AI Generativa com foco em produtividade': 'uso_pago_ia_gen_produtividade',
    '4.j.4 A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade': 'uso_pago_ia_gen_empresa_paga',
    '4.j.5 Uso soluções do tipo Copilot': 'uso_copilot',
    '5.a_objetivo_na_area_de_dados': 'obj_area_dados',
    '5.b_oportunidade_buscada': 'oportunidade_almejada',
    '5.c_tempo_em_busca_de_oportunidade': 'tempo_busca_oport_area_dados',
    '5.d_experiencia_em_processos_seletivos': 'busca_empreg_area_dados',
    '6.a_rotina_como_de': 'rotina_engenheiro_dados',
    '6.a.1_Desenvolvo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.': 'pipeline_dados_ling_prog',
    "6.a.2_Realizo construções de ETL's em ferramentas como Pentaho, Talend, Dataflow etc.": 'etl_pentaho_talent_etc',
    '6.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.': 'relatorios_sql_exportados',
    '6.a.4_Atuo na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.': 'integracao_dados_plataformas_dados',
    '6.a.5_Modelo soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.': 'criacao_componentes_ingestao_dados',
    '6.a.6_Desenvolvo/cuido da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.': 'manutencao_criacao_rep_dados_streaming',
    '6.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.': 'modelagem_dados_dw_data_mart',
    '6.a.8_Cuido da qualidade dos dados, metadados e dicionário de dados.': 'quali_dados_dicionario',
    '6.a.9_Nenhuma das opções listadas refletem meu dia a dia.': 'nenhuma_opcao_engenheiro_dados',
    '6.b_ferramentas_etl_de': 'ferramenta_etl_utilizado_engenheiro_dados',
    '6.b.1_Scripts Python': 'script_python',
    '6.b.2_SQL & Stored Procedures': 'sql_stored_procedures',
    '6.b.3_Apache Airflow': 'apache_airflow',
    '6.b.4_Apache NiFi': 'apache_nifi',
    '6.b.5_Luigi': 'luigi',
    '6.b.6_AWS Glue': 'aws_glue',
    '6.b.7_Talend': 'talend',
    '6.b.8_Pentaho': 'engenheiro_dados_pentaho',
    '6.b.9_Alteryx': 'engenheiro_dados_alteryx',
    '6.b.10_Stitch': 'stitch',
    '6.b.11_Fivetran': 'fivetran',
    '6.b.12_Google Dataflow': 'google_dataflow',
    '6.b.13_Oracle Data Integrator': 'oracle_data_integrator',
    '6.b.14_IBM DataStage': 'ibm_data_stage',
    '6.b.15_SAP BW ETL': 'sap_bw_etl',
    '6.b.16_SQL Server Integration Services (SSIS)': 'SSIS',
    '6.b.17_SAS Data Integration': 'sas_data_integration',
    '6.b.18_Qlik Sense': 'qlik_sense',
    '6.b.19_Knime': 'knime',
    '6.b.20_Databricks': 'engenheiro_dados_databricks',
    '6.b.21_Não utilizo ferramentas de ETL': 'engenheiro_dados_sem_ferramenta_etl',
    '6.c_possui_data_lake': 'empresa_possui_datalake',
    '6.d_tecnologia_data_lake': 'tecnologia_utilizada_datalake',
    '6.e_possui_data_warehouse': 'empresa_possui_dw',
    '6.f_tecnologia_data_warehouse': 'tecnologia_utilizada_dw',
    '6.g_ferramentas_de_qualidade_de_dados_(dia_a_dia)': 'ferramentas_meta_dados_qualidade_dados_trabalho',
    '6.h.1_Desenvolvendo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.': 'pipelines_com_codigo',
    '6.h.2_Realizando construções de ETL\\s em ferramentas como Pentaho, Talend, Dataflow etc.': 'etl_com_ferramentas_bi',
    '6.h.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.': 'exportacao_rel_para_area_neg',
    '6.h.4_Atuando na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.': 'integracao_fonte_dados_diferentes_tmp_gasto',
    '6.h.5_Modelando soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.': 'modelagem_arquitetura_dados_tmp_gasto',
    '6.h.6_Desenvolvendo/cuidando da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.': 'dados_streaming_data_lake_lakehouse',
    '6.h.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.': 'modelagem_dados_dw_data_mart',
    '6.h.8_Cuidando da qualidade dos dados, metadados e dicionário de dados.': 'qualidade_dados_tmp_gasto',
    '6.h.9_Nenhuma das opções listadas refletem meu dia a dia.': 'nenhuma_das_opcoes_tmp_gasto',
    '6.h_maior_tempo_gasto_como_de': 'maior_parte_tempo_gasta',
    '7.a.1_Processo e analiso dados utilizando linguagens de programação como Python, R etc.': 'analise_processamento_linguagem_prog',
    '7.a.2_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.': 'dashboard_ferramentas_bi',
    '7.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.': 'exportacao_rel_area_neg_analista_dados',
    '7.a.4_Utilizo API\\s para extrair dados e complementar minhas análises.': 'extracao_dados_api',
    '7.a.5_Realizo experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.': 'estudo_experimentos_util_estatistica',
    '7.a.6_Desenvolvo/cuido da manutenção de ETL\\s utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.': 'manutencao_etl_utilizando_ferramenta_etl_analista',
    '7.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts etc.': 'modelagem_dw_data_mart_analista',
    '7.a.8_Desenvolvo/cuido da manutenção de planilhas para atender as áreas de negócio.': 'manutencao_planilha_area_negocio',
    '7.a.9_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.': 'utilizacao_ferramenta_avancada_estatistica',
    '7.a.10_Nenhuma das opções listadas refletem meu dia a dia.': 'nenhuma_das_opcoes_analista',
    '7.a_rotina_como_da': 'analise_dados_rotina',
    '7.b.1_Scripts Python': 'python_analista',
    '7.b.2_SQL & Stored Procedures': 'stored_procedures_analista',
    '7.b.3_Apache Airflow': 'apache_airflow_analista',
    '7.b.4_Apache NiFi': 'apache_nifi_analista',
    '7.b.5_Luigi': 'luigi_analista',
    '7.b.6_AWS Glue': 'aws_glue_analista',
    '7.b.7_Talend': 'talent_analista',
    '7.b.8_Pentaho': 'pentaho_analista',
    '7.b.9_Alteryx': 'alteryx_analista',
    '7.b.10_Stitch': 'stitch_analista',
    '7.b.11_Fivetran': 'fivetran_analista',
    '7.b.12_Google Dataflow': 'google_dataflow_analista',
    '7.b.13_Oracle Data Integrator': 'oracle_data_integrator_analista',
    '7.b.14_IBM DataStage': 'ibm_data_stage_analista',
    '7.b.15_SAP BW ETL': 'sap_bw_etl_analista',
    '7.b.16_SQL Server Integration Services (SSIS)': 'ssis_analista',
    '7.b.17_SAS Data Integration': 'sas_data_integration_analista',
    '7.b.18_Qlik Sense': 'qlik_sense_analista',
    '7.b.19_Knime': 'knime_analista',
    '7.b.20_Databricks': 'databricks_analista',
    '7.b.21_Não utilizo ferramentas de ETL': 'nao_utilizo_ferramentas_etl_analista',
    '7.b_ferramentas_etl_da': 'ferramental_analista_dados',
    '7.c.1_Ferramentas de AutoML como H2O.ai, Data Robot, BigML etc.': 'ferramentas_auto_ml',
    '7.c.2_"Point and Click" Analytics como Alteryx, Knime, Rapidminer etc.': 'point_click',
    '7.c.3_Product metricts & Insights como Mixpanel, Amplitude, Adobe Analytics.': 'product_metrics_insight',
    '7.c.4_Ferramentas de análise dentro de ferramentas de CRM como Salesforce Einstein Anaytics ou Zendesk dashboards.': 'ferramentas_analise_dados_dentro_crm',
    '7.c.5_Minha empresa não utiliza essas ferramentas.': 'empresa_nao_utiliza_ferramenta_autonomia',
    '7.c.6_Não sei informar.': 'nao_sei_informar_ferramenta_autonomia_area_neg',
    '7.c_ferramentas_autonomia_area_de_negocios': 'ferramentas_empresa_usadas_mais_autonomia_area_neg',
    '7.d.1_Processando e analisando dados utilizando linguagens de programação como Python, R etc.': 'dashboards_script_analista',
    '7.d.2_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.': 'dashboards_ferramenta_bi_analista',
    '7.d.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.': 'extracao_rel_sql_analista',
    "7.d.4_Utilizando API's para extrair dados e complementar minhas análises.": 'extracao_api_analista',
    '7.d.5_Realizando experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.': 'estudo_uso_estatistica_analista',
    "7.d.6_Desenvolvendo/cuidando da manutenção de ETL's utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": 'criacao_manutencao_etl_analista',
    '7.d.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.': 'modelagem_dados_dw_data_mart_analista',
    '7.d.8_Desenvolvendo/cuidando da manutenção de planilhas para atender as áreas de negócio.': 'manutencao_planilhas_analista',
    '7.d.9_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.': 'ferramentas_avanc_estatistica_analista',
    '7.d.10_Nenhuma das opções listadas refletem meu dia a dia.': 'nenhuma_opcao_listada_atv_analista',
    '7.d_maior_tempo_gasto_como_da': 'maior_parte_tempo_gasto_analista',
    '8.a.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.': 'uso_adhoc_modelo_preditivo',
    '8.a.2_Sou responsável pela coleta e limpeza dos dados que uso para análise e modelagem.': 'coleta_limpeza_modelagem_analise',
    '8.a.3_Sou responsável por entrar em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.': 'analise_requisitos_area_negocio',
    '8.a.4_Desenvolvo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).': 'dev_modelo_ml',
    '8.a.5_Sou responsável por colocar modelos em produção, criar os pipelines de dados, APIs de consumo e monitoramento.': 'deploy_modelo_prod_criacao_api_monitoramento',
    '8.a.6_Cuido da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.': 'manutencao_modelo_ml_producao_melhorias',
    '8.a.7_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc': 'construcao_dashboard_ferramenta_bi_analista',
    '8.a.8_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.': 'ferramenta_avanc_estatistica_analise_ajust_modelo',
    '8.a.9_Crio e dou manutenção em ETLs, DAGs e automações de pipelines de dados.': 'criacao_manun_etl_dags_automacoes',
    '8.a.10_Crio e gerencio soluções de Feature Store e cultura de MLOps.': 'criacao_gerencia_feature_store_mlops',
    '8.a.11_Sou responsável por criar e manter a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)': 'infra_modelos_rodam_em_clusters_api_etc',
    "8.a.12_Treino e aplico LLM's para solucionar problemas de negócio.": 'criacao_treinamento_llm',
    '8.a_rotina_como_ds': 'rotina_trabalho_cientista',
    '8.b.1_Utilizo modelos de regressão (linear, logística, GLM).': 'regressao_logistica_glm',
    '8.b.2_Utilizo redes neurais ou modelos baseados em árvore para criar modelos de classificação.': 'redes_neurais_odelo_arvore',
    '8.b.3_Desenvolvo sistemas de recomendação (RecSys).': 'sistema_recomendacao',
    '8.b.4_Utilizo métodos estatísticos Bayesianos para analisar dados.': 'metodo_estatistico_bayesiano_analise',
    '8.b.5_Utilizo técnicas de NLP (Natural Language Processing) para análisar dados não-estruturados.': 'NLP_dados_nao_estruturados',
    '8.b.6_Utilizo métodos estatísticos clássicos (Testes de hipótese, análise multivariada, sobrevivência, dados longitudinais, inferência estatistica) para analisar dados.': 'metodos_estatisticos_classicos',
    '8.b.7_Utilizo cadeias de Markov ou HMM\\s para realizar análises de dados.': 'cadeias_markov_hmm_analise_dados',
    '8.b.8_Desenvolvo técnicas de Clusterização (K-means, Spectral, DBScan etc).': 'tecnicas_clusterizacao',
    '8.b.9_Realizo previsões através de modelos de Séries Temporais (Time Series).': 'modelos_preditivos',
    '8.b.10_Utilizo modelos de Reinforcement Learning (aprendizado por reforço).': 'aprendizado_reforco',
    '8.b.11_Utilizo modelos de Machine Learning para detecção de fraude.': 'ml_deteccao_fraude',
    '8.b.12_Utilizo métodos de Visão Computacional.': 'visao_computacional',
    '8.b.13_Utilizo modelos de Detecção de Churn.': 'modelo_deteccao_churn',
    "8.b.14_Utilizo LLM's para solucionar problemas de negócio.": 'llm_cientista_dados_uso',
    '8.b_tecnicas_e_metodos_ds': 'tecnica_metodo_utilizado_cientista_trabalho',
    '8.c.1_Ferramentas de BI (PowerBI, Looker, Tableau, Qlik etc).': 'ferramenta_bi_cientista',
    '8.c.2_Planilhas (Excel, Google Sheets etc).': 'planilha_excel_cientista',
    '8.c.3_Ambientes de desenvolvimento local (R-studio, JupyterLab, Anaconda).': 'ambiente_local_jupyter_anaconda',
    '8.c.4_Ambientes de desenvolvimento na nuvem (Google Colab, AWS Sagemaker, Kaggle Notebooks etc).': 'dev_nuvem',
    '8.c.5_Ferramentas de AutoML (Datarobot, H2O, Auto-Keras etc).': 'auto_ml_cientista',
    '8.c.6_Ferramentas de ETL (Apache Airflow, NiFi, Stitch, Fivetran, Pentaho etc).': 'ferramenta_etl_cientista',
    '8.c.7_Plataformas de Machine Learning (TensorFlow, Azure Machine Learning, Kubeflow etc).': 'plataforma_ml_cientista',
    '8.c.8_Feature Store (Feast, Hopsworks, AWS Feature Store, Databricks Feature Store etc).': 'feature_store_cientista',
    '8.c.9_Sistemas de controle de versão (Github, DVC, Neptune, Gitlab etc).': 'controle_de_versao_cientista',
    '8.c.10_Plataformas de Data Apps (Streamlit, Shiny, Plotly Dash etc).': 'data_apps_cientista',
    '8.c.11_Ferramentas de estatística avançada como SPSS, SAS, etc.': 'ferramenta_estatistica_avanc_cientista',
    '8.c_tecnologias_ds': 'tecnologias_diarias_cientista',
    '8.d.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.': 'ad_hoc_hipoteses_tmp_gasto_cientista',
    '8.d.2_Coletando e limpando dos dados que uso para análise e modelagem.': 'coleta_limpeza_cientista',
    '8.d.3_Entrando em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.': 'analise_requisitos_tmp_gasto_cientista',
    '8.d.4_Desenvolvendo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).': 'dev_ml_prod_tmp_gasto_cientista',
    '8.d.5_Colocando modelos em produção, criando os pipelines de dados, APIs de consumo e monitoramento.': 'manutencao_ml_prod_tmp_gasto_cientista',
    '8.d.6_Cuidando da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.': 'manutencao_ml_em_prod_monitoramento_ajuste_tmp_gasto_cientista',
    '8.d.7_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc.': 'dashboards_ferramenta_bi_tmp_gasto_cientista',
    '8.d.8_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.': 'ferramenta_avancadas_estatistica_analista_tmp_gasto_cientista',
    '8.d.9_Criando e dando manutenção em ETLs, DAGs e automações de pipelines de dados.': 'manutencao_etl_dag_tmp_gasto_cientista',
    '8.d.10_Criando e gerenciando soluções de Feature Store e cultura de MLOps.': 'feature_store_ml_ops_tmp_gasto_cientista',
    '8.d.11_Criando e mantendo a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)': 'infra_modelo_clusters_servidores_tmp_gasto_cientista',
    "8.d.12_Treinando e aplicando LLM's para solucionar problemas de negócio.": 'LLM_tmp_gasto_cientista',
    '8.d_maior_tempo_gasto_como_ds': 'tempo_gasto_cientista'
}

In [17]:
df_state_data_2025 = ler_df_csv(sessao, caminho_2025)

In [18]:
df_state_data_2025 = renomear_colunas(df_state_data_2025, novos_nomes_colunas_2025)
df_state_data_2025 = processar_dataframe(df_state_data_2025)

In [22]:
exportar_df_para_csv(df_state_data_2025, "bronze_dw_state_data_2025.csv")